# Day 4
# Graph neural networks, and putting the week together

**Goal:** turn the cleaned complexes into PyTorch Geometric graphs, train the RexGCN model that
this project actually uses, measure it honestly with cross-validation, and close the workshop.

**Source paper:** [Redox potential prediction of Fe(ii)/Fe(iii) complexes: a density functional theory and graph neural network approach](https://pubs.rsc.org/dd/article/5/3/1098/311651/Redox-potential-prediction-of-Fe-ii-Fe-iii )

## Learning objectives

- Convert RDKit molecules into PyG `Data` objects with node, edge, and graph-level features.
- Read a real message-passing layer and say what each part of it does.
- Train a GNN on GPU with a learning-rate schedule and early stopping.
- Use k-fold cross-validation to report a performance range rather than a single number.
- Recognise what the model does not use, and what it would take to add it.
- Present the week's work as a five-minute talk.

---


## Recap of Days 1 to 3

- **Day 1** set up the NERSC environment and the chemistry. Fe(ii)/Fe(iii) redox potentials,
  the tmQM structures, and the cleaning that produced `output/data_cleaned.pkl`: 1546
  complexes with an RDKit molecule, a SMILES string, and a measured potential each.
- **Day 2** was exploratory. Distributions and moments of the target, correlations between
  features, PCA and K-Means on the tabular features, and the vocabulary of machine learning:
  train, validation and test, data leakage, and why the deployed model is retrained on
  everything.
- **Day 3** built two descriptions of the same complexes. First the numerical ones: Random
  Forest and Gaussian Process baselines on tabular features, Morgan fingerprints, PCA of
  fingerprint space, and ligand classification. Then the structural one: a molecule as a
  graph, with atoms as nodes, bonds as edges, node and edge features, and an adjacency
  matrix.

Today picks up exactly where Day 3 section 6.10 stopped. The baselines from Day 3 Part 1 are
the numbers to beat.

---


### Set `REPO_ROOT`

Every notebook in this workshop locates the repository the same way: by walking up
from the notebook's own directory until it finds one containing both `data/` and
`notebooks/`.

Run this cell before any other code cell, and check that the printed path is your clone.


In [ ]:
# --- Set REPO_ROOT --------------------------------------------------------
# Locate the repository root by searching upward from this notebook's directory.
from pathlib import Path


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for d in (start, *start.parents):
        if (d / "data").is_dir() and (d / "notebooks").is_dir():
            return d
    raise FileNotFoundError(f"Repo root not found above {start}")


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "output"
FIG_DIR = OUTPUT_DIR / "figures"

OUTPUT_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

print("Repo root: ", REPO_ROOT)
print("Data dir:  ", DATA_DIR)
print("Output dir:", OUTPUT_DIR)
print("Figure dir:", FIG_DIR)


## Part 1 - Theory: Message passing and graph-level prediction

### 1.1 Molecules as graphs

A molecule is represented as a graph $G = (V, E)$ where $V$ is the set of nodes (atoms) and $E$ is the set of edges (bonds or proximity links). Each node $v_i$ has an initial feature vector

$$\mathbf{h}_i^{(0)} \in \mathbb{R}^{d_{\text{node}}}$$

Edges may also carry features (bond order, distance, ring flags).

Common node features: atomic number, electronegativity, formal charge, hybridization, 3D coordinates.

Common edge features: interatomic distance, bond order, whether the edge is in a ring.

### 1.2 Message Passing Neural Networks (MPNNs)

A generic MPNN layer performs a message aggregation followed by an update:

$$\mathbf{h}_i^{(l+1)} = \mathrm{UPDATE}\!\left(\mathbf{h}_i^{(l)}, \mathrm{AGGREGATE}\!\left(\{\mathbf{h}_j^{(l)} : j \in \mathcal{N}(i)\}\right)\right)$$

where $\mathcal{N}(i)$ denotes neighbors of node $i$.

A common concrete form is the GCN layer (Kipf & Welling, 2017):

$$\mathbf{h}_i^{(l+1)} = \sigma\!\left(\sum_{j \in \mathcal{N}(i) \cup \{i\}} \frac{1}{\sqrt{|\mathcal{N}(i)| \cdot |\mathcal{N}(j)|}} W^{(l)} \mathbf{h}_j^{(l)}\right)$$

Readout (graph-level pooling) turns node embeddings into a single vector used for regression:

$$\widehat{y} = \mathrm{MLP}\!\left(\mathrm{READOUT}\!\left(\{\mathbf{h}_i^{(L)} : v_i \in V\}\right)\right)$$

Common READOUTs: mean, sum, max, or attention-weighted pooling.

---


## Part 2 - Building the data pipeline

Day 3 drew a molecule as a graph with NetworkX. NetworkX is fine for looking at, but a model
needs tensors, so from here the graphs are PyTorch Geometric `Data` objects. The code in this
part is the production pipeline from the paper, not a teaching mock-up.


In [ ]:
# Imports and GPU check.
import os
import random
import math
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, AutoMinorLocator
from rdkit import Chem
from IPython.display import display
from sklearn.metrics import r2_score, mean_squared_error

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# GPU check
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    try:
        props = torch.cuda.get_device_properties(0)
        # property is `total_memory` in recent PyTorch builds
        print('GPU name:', props.name)
        print('Total memory (bytes):', props.total_memory)
    except Exception:
        pass

# Filenames we keep the same as the course standard
BEST_CHECKPOINT = 'best_gnn_model.pt'
FINAL_MODEL = 'fe_redox_gnn_final.pt'
TRAIN_CURVES = 'gnn_training_curves.png'
TEST_PARITY = 'gnn_test_parity.png'
FULL_COMPARISON = 'full_model_comparison.png'
FINAL_RESULTS = 'final_results.csv'

# FIG_DIR comes from the REPO_ROOT cell; model checkpoints get their own directory
MODEL_DIR = OUTPUT_DIR / 'saved_models'
MODEL_DIR.mkdir(exist_ok=True)



### 2.1 Node, edge, and graph features

Day 3 section 6.3 named the three levels a molecular graph can carry. This is what the
pipeline below actually puts in each of them.

**Node features: 119 numbers per atom.** The row of $X$ belonging to one atom is the
concatenation of

| Block | Size | Contents |
|---|---:|---|
| scaled atomic number | 1 | $(Z-1)/91$, so hydrogen is 0 and uranium is 1 |
| element one-hot | 92 | a 1 in the slot for that element |
| degree one-hot | 8 | number of bonded neighbors, 0 to 7 |
| formal charge | 1 | as an integer |
| radical electrons | 1 | count |
| hybridization one-hot | 7 | S, SP, SP2, SP3, SP3D, SP3D2, unspecified |
| aromaticity | 1 | 0 or 1 |
| attached hydrogens one-hot | 5 | 0 to 4 |
| scaled atomic mass | 1 | shifted and divided so the range is roughly 0 to 1 |
| scaled van der Waals radius | 1 | same treatment |
| scaled covalent radius | 1 | same treatment |

**Edge features: 10 numbers per bond.** Four bond-type flags (single, double, triple,
dative), conjugation, ring membership, and a four-slot stereo one-hot. Dative bonds matter
here: the metal-ligand bonds in these complexes are written as dative in the SMILES.

**Graph-level fields.** The `Data` object also carries a Morgan fingerprint (`morgan_fp`),
the iron formal charge (`metal_charge`), a one-hot of which elements are bonded to the iron
(`metal_neigh_symbols`), the summed charge of those neighbors (`metal_neigh_charge`), the
target `y`, and `z` and `pos`: atomic numbers and 3D coordinates.

**What the model in Part 3 uses:** node features and edge features only. The graph-level
fields ride along in the dataset and are ignored. Section 3.3 comes back to this, and it is
the subject of the first exercise.

Scaling note: the continuous entries are already scaled into roughly $[0, 1]$ inside the
feature builder, so there is no separate `StandardScaler` step here as there was for the
tabular baselines on Day 3.


### 2.2 Loading the dataset

The same cleaned dataset as Day 3. The graph functions below expect the target in a column
called `y`, and refer to the dataframe as `final_df`, so both names are set up here.


In [ ]:
# The cleaned dataset from Day 1 and 2: 1546 complexes, each with an RDKit mol that carries
# a 3D conformer. The graph pipeline needs those coordinates, the SMILES, and the target.

CLEANED_PATH = OUTPUT_DIR / 'data_cleaned.pkl'

final_df = pd.read_pickle(CLEANED_PATH)
final_df = final_df.reset_index()          # csd_code is stored as the index in this file

# GenGraphs and every training cell below read the target from a column named 'y'
final_df['y'] = final_df['redox_pot']

print('Loaded', len(final_df), 'complexes from', CLEANED_PATH)
print('Molecules carrying 3D coordinates:',
      sum(1 for m in final_df['mol'] if m is not None and m.GetNumConformers() > 0))
display(final_df[['csd_code', 'smiles', 'y']].head())


### 2.3 Graph generation

`GenGraphs` walks a dataframe and returns a list of PyG `Data` objects, one per complex. It
is the single place where chemistry becomes tensors, so it is worth reading slowly.


In [ ]:
# Turns a dataframe of RDKit molecules into a list of PyTorch Geometric Data objects.
# One Data object per complex; the feature layout is the one tabulated in 2.1.

# Helper functions

from rdkit import Chem
from rdkit.Chem import AllChem, rdchem, rdFingerprintGenerator


def get_num_Hs(atom):
    """
    Returns the number of hydrogen atoms bonded to the given atom.
    """
    num_Hs = 0
    for neighbor in atom.GetNeighbors():
        if neighbor.GetAtomicNum() == 1:  # Atomic number 1 corresponds to hydrogen
            num_Hs += 1
    return num_Hs

    
def compute_morgan_fingerprint(mol, radius=3, fpSize=2048):
    mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=radius,fpSize=fpSize)
    morgan_fp = mfpgen.GetFingerprint(mol) # Get the fingerprint object
    # Convert ExplicitBitVect to NumPy array
    np_fp = np.array(list(morgan_fp))#, dtype=float)
    return np_fp # Return the NumPy array

# Creating dataset

# Define constants for hybridizations and stereoisomers
hybridizations = [
            Chem.rdchem.HybridizationType.S,
            Chem.rdchem.HybridizationType.SP,
            Chem.rdchem.HybridizationType.SP2,
            Chem.rdchem.HybridizationType.SP3,
            Chem.rdchem.HybridizationType.SP3D,
            Chem.rdchem.HybridizationType.SP3D2,
            Chem.rdchem.HybridizationType.UNSPECIFIED
        ]

stereos = [rdchem.BondStereo.STEREONONE, rdchem.BondStereo.STEREOANY,
           rdchem.BondStereo.STEREOZ, rdchem.BondStereo.STEREOE]

# Define periodic table for atomic properties
periodic_table = Chem.GetPeriodicTable()


# The main entry point. mfp_mask selects which fingerprint bits to keep as a graph-level
# feature; passing None (what this notebook does) falls back to a 64-bit fingerprint.
def GenGraphs(df, mfp_mask=None, mfp_radius = 3, mfp_bits = 2048):
    data_list = []
    
    # Mols
    mol_list = df['mol'].tolist()
    
    # Graph features
    #homoE_list = df['HOMO_Energy'].tolist()
    #lumoE_list = df['LUMO_Energy'].tolist()
    #polarize_list = df['Polarizability'].tolist()
    #electronic_E_list = df['Electronic_E'].tolist()
    #dispersion_E_list = df['Dispersion_E'].tolist()

    smiles_list = df['smiles'].tolist()
    
    # target
    has_target = 'y' in df.columns
    if has_target:
        y_list = df['y'].tolist()

    
    for idx, mol in enumerate(mol_list):
        
        if has_target:
            y_val = y_list[idx]
            y = torch.tensor(y_val, dtype=torch.float).view(-1, 1)
        else:
            y = None # Pass None if no target exists

        #homoE = torch.tensor([homoE_list[idx]], dtype=torch.float)
        #lumoE = torch.tensor([lumoE_list[idx]], dtype=torch.float)
        #polarize = torch.tensor([polarize_list[idx]], dtype=torch.float)
        #electronic_E = torch.tensor([electronic_E_list[idx]], dtype=torch.float)
        #dispersion_E = torch.tensor([dispersion_E_list[idx]], dtype=torch.float)
    

        # The commented-out lines above and below are other graph-level features the
        # production pipeline can attach (DFT energies, polarizability). They are left
        # in place to show where extra graph-level information would go.
        smiles = smiles_list[idx]
        
        # Compute Morgan fingerprint
        if mfp_mask is None:
            #mfp_mask = np.ones(n_features, dtype=bool)
            mfp_bits = 64
            mfp_mask = np.ones(mfp_bits, dtype=bool)
        morgan_fp = compute_morgan_fingerprint(mol, radius=mfp_radius, fpSize=mfp_bits)
        #n_features = morgan_fp.shape[0]
        
        morgan_fp = morgan_fp[mfp_mask]
        #morgan_fp_tensor = torch.tensor(list(morgan_fp), dtype=torch.float)
        morgan_fp_tensor = torch.tensor(list(morgan_fp), dtype=torch.float).view(1, -1) 
        #print(sum(morgan_fp_tensor), len(morgan_fp_tensor))

        #mol = Chem.MolFromSmiles(smiles)
        #if mol is None:
        #    continue  # Skip invalid SMILES

        

        # Add hydrogens
        
        #mol = Chem.AddHs(mol)
        
        # Sanity filters, repeated here so the function is safe on any dataframe:
        # skip anything with an implausible coordination number or iron oxidation state.
        atoms = mol.GetAtoms()
        max_valence = 0
        for _, atom in enumerate(atoms):
            max_valence = atom.GetDegree() if atom.GetDegree() > max_valence else max_valence
            if atom.GetSymbol() == 'Fe':
                Fe_charge = torch.tensor([atom.GetFormalCharge()], dtype=torch.float)
        
        if max_valence > 7:
            continue
        if Fe_charge > 3 or Fe_charge < 0:
            continue

        # 3D coordinates, one row per atom. The GCN below never looks at pos, but SchNet
        # and DimeNet++ in Part 7 are built entirely on interatomic distances and angles.
        conformer = mol.GetConformer() # Get the default conformer
        pos = [conformer.GetAtomPosition(atom.GetIdx()) for atom in mol.GetAtoms()]
        pos = torch.tensor(pos, dtype=torch.float)
        
        # ---- NODE FEATURES ----
        # xs collects one feature vector per atom; they are stacked into X at the end.
        xs = []
        Fe_neigh_symbols = [0] * 92
        Fe_neigh_charge = 0
        for atom in atoms:
            atomic_num_scaled = float((atom.GetAtomicNum() - 1) / 91)  # H: 1, U: 92
            symbol = [0] * 92 #len(periodic_table.GetMaxAtomicNumber())
            try:
                symbol[periodic_table.GetAtomicNumber(atom.GetSymbol()) - 1] = 1.
            except IndexError:
                print(f"Error: Unknown atom symbol {atom.GetSymbol()} in {smiles}")
                break # Stop processing atom
            
            if atom.GetSymbol() == 'Fe':
                Fe_charge = torch.tensor([atom.GetFormalCharge()], dtype=torch.float)
                # Get neighbors of the Fe atom
                neighbors = atom.GetNeighbors()
                for neighbor in neighbors:
                    # Get the symbol of the neighbor atom
                    neigh_symbol = neighbor.GetSymbol()
                    # Append the symbol to the list
                    try:
                        Fe_neigh_symbols[periodic_table.GetAtomicNumber(neigh_symbol) - 1] = 1.
                    except IndexError:
                        continue
                    Fe_neigh_charge += neighbor.GetFormalCharge()
                
                Fe_neigh_charge = torch.tensor([Fe_neigh_charge], dtype=torch.float)
                Fe_neigh_symbols = torch.tensor(Fe_neigh_symbols, dtype=torch.float)
            
            #print(atom.GetSymbol(), atom.GetFormalCharge(), atom.GetDegree(), atom.GetHybridization(), atom.GetTotalNumHs())
            # one-hot of the number of bonded neighbours (the graph degree from Day 3 6.8)
            valance = [0.] * 8
            valance[atom.GetDegree()] = 1.

            formal_charge = atom.GetFormalCharge()   # [-3, -2, -1, 0, 1, 2, 3]
            radical_electrons = atom.GetNumRadicalElectrons()
            
            hybridization = [0.] * len(hybridizations)
            if atom.GetHybridization() in hybridizations:
                hybridization[hybridizations.index(atom.GetHybridization())] = 1.
                
            aromaticity = 1. if atom.GetIsAromatic() else 0.
            hydrogens = [0.] * 5
            try:
                hydrogens[get_num_Hs(atom)] = 1.
            except IndexError: # Handle cases with > 4 explicit Hs if possible
                 print(f"Warning: Atom {atom.GetIdx()} in mol {idx} has unexpected num Hs {get_num_Hs(atom)}.")
            #print(atom.GetSymbol(), hydrogens)
            
            '''chirality = 1. if atom.HasProp('_ChiralityPossible') else 0.
            chirality_type = [0.] * 2
            if atom.HasProp('_CIPCode'):
                try:
                    chirality_type[['R', 'S'].index(atom.GetProp('_CIPCode'))] = 1.
                except ValueError:
                    continue'''

            # Continuous properties, shifted and divided so each lands roughly in [0, 1].
            # Without this the atomic mass would dominate every one-hot slot around it.
            atomic_mass = float((periodic_table.GetAtomicWeight(atom.GetAtomicNum()) - 1.008) / 237.021)
            vdw_radius = float((periodic_table.GetRvdw(atom.GetAtomicNum()) - 1.2) / 1.35)
            covalent_radius = float((periodic_table.GetRcovalent(atom.GetAtomicNum()) - 0.23) / 1.71)

            x = torch.tensor([atomic_num_scaled] + symbol + valance +
                             [formal_charge] + [radical_electrons] +
                             hybridization + [aromaticity] +
                             hydrogens + #[chirality] + chirality_type +
                             [atomic_mass] +
                             [vdw_radius] + [covalent_radius], dtype=torch.float)
            xs.append(x)

        # check mol
        if len(xs) != mol.GetNumAtoms():
             print(f"Skipping {smiles}: Mismatch between RDKit atoms and generated features.")
             continue
        
        x = torch.stack(xs, dim=0) if len(xs) > 0 else torch.tensor([], dtype=torch.float)

        atomic_numbers = [atom.GetAtomicNum() for atom in mol.GetAtoms()]
        z = torch.tensor(atomic_numbers, dtype=torch.long) if atomic_numbers else torch.tensor([], dtype=torch.long)

        # --- Assertion Checks --- #
        num_atoms_pos = pos.shape[0]
        num_atoms_z = z.shape[0]
        assert num_atoms_pos == num_atoms_z, \
                                f"CRITICAL MISMATCH: num_atoms_pos ({num_atoms_pos}) != num_atoms_z ({num_atoms_z}) for molecule {idx}" # Replace mol_identifier with SMILES or I

        # ---- EDGES ----
        # Each bond is written twice, (start, end) and (end, start), because PyG expects
        # a directed edge list and an undirected bond is two directed edges. edge_attrs
        # gets the same feature vector twice for the same reason.
        row, col, edge_attrs = [], [], []

        for bond in mol.GetBonds():
            start = bond.GetBeginAtomIdx()
            end = bond.GetEndAtomIdx()

            row += [start, end]
            col += [end, start]

            bond_type = bond.GetBondType()
            single = 1. if bond_type == Chem.rdchem.BondType.SINGLE else 0.
            double = 1. if bond_type == Chem.rdchem.BondType.DOUBLE else 0.
            triple = 1. if bond_type == Chem.rdchem.BondType.TRIPLE else 0.
            dative = 1. if bond_type == Chem.rdchem.BondType.DATIVE else 0.

            conjugated = 1. if bond.GetIsConjugated() else 0.
            ring_bond = 1. if bond.IsInRing() else 0.

            stereo = [0.] * len(stereos)
            if bond.GetStereo() in stereos:
                stereo[stereos.index(bond.GetStereo())] = 1.

            edge_attr = torch.tensor([single, double, triple, dative,
                                      conjugated, ring_bond] + stereo, dtype=torch.float)
            
            edge_attrs += [edge_attr, edge_attr]

        edge_index = torch.tensor([row, col], dtype=torch.long) if len(row) > 0 else torch.tensor([], dtype=torch.long).view(2, -1)
        edge_attr = torch.stack(edge_attrs, dim=0) if len(edge_attrs) > 0 else torch.tensor([], dtype=torch.float)

        # Debugging: Print shapes and contents
        #print(f"edge_index shape: {edge_index.shape}")
        #print(f"edge_attr shape: {edge_attr.shape}")
        
        # Sort indices.
        # Sort edges by (source, target) so the ordering is deterministic between runs.
        if edge_index.numel() > 0:
            perm = (edge_index[0] * x.size(0) + edge_index[1]).argsort()
            edge_index, edge_attr = edge_index[:, perm], edge_attr[perm]

        # Everything assembled into one graph. x and edge_attr are what the model reads;
        # morgan_fp, metal_charge, metal_neigh_* and smiles are graph-level and unused
        # by RexGCN as written (see 3.3).
        data = Data(x=x, z=z, pos=pos, edge_index=edge_index, edge_attr=edge_attr, y=y,
                    metal_charge=Fe_charge, metal_neigh_symbols=Fe_neigh_symbols, metal_neigh_charge=Fe_neigh_charge,   # none of these are used currently as graph features
                    morgan_fp=morgan_fp_tensor, smiles=smiles)
                    #electronic_E=electronic_E, dispersion_E=dispersion_E, 
                    #pca_0=pca_0, pca_1=pca_1, tsne_0=tsne_0, tsne_1=tsne_1, smiles=smiles) homoE= homoE, lumoE=lumoE, polarize=polarize, )
        

        data_list.append(data)
        

    return data_list

### 2.4 One graph, inspected

Before building thousands, build five and look at one.


In [ ]:
# A small batch first: check the shapes against the tables in 2.1 before committing to the
# full dataset. The cross-validation and final-split cells below each build their own graphs.

demo_graphs = GenGraphs(final_df.head(5))

d = demo_graphs[0]
print(d)
print()
print('nodes            :', d.x.size(0))
print('node feature dim :', d.x.size(1))
print('directed edges   :', d.edge_index.size(1))
print('edge feature dim :', d.edge_attr.size(1))
print('graph-level      : y', tuple(d.y.shape),
      '| morgan_fp', tuple(d.morgan_fp.shape),
      '| metal_charge', tuple(d.metal_charge.shape),
      '| metal_neigh_symbols', tuple(d.metal_neigh_symbols.shape))
print('3D coordinates   :', tuple(d.pos.shape))
print()
print('first atom feature vector (first 12 entries):')
print(d.x[0][:12])



**Mentor checkpoint 9**: after the graph pipeline

- Do the printed node and edge feature dimensions match the tables in 2.1? Which block would
  you have to change to add a new atom property?
- Why is the number of directed edges twice the number of bonds?
- Which fields of the `Data` object are node-level, which are edge-level, and which belong to
  the graph as a whole?
- What happens to a complex whose molecule has no 3D conformer?

Proceed only after confirmation.

---


## Part 3 - GNN model architecture

Two models follow. The first is a small demonstration written for this course, short enough
to read in one sitting. The second is the model this project actually uses.

### 3.1 A minimal GNN

Every graph neural network for property prediction has the same four stages:

1. **Embed.** A linear layer lifts the raw node features to a working width, `hidden_dim`.
2. **Message passing.** Each layer replaces every atom's vector with a function of its own
   vector and its neighbors'. Stack $L$ layers and information travels $L$ bonds, exactly the
   rounds sketched in Day 3 section 6.10.
3. **Readout, or pooling.** The per-atom vectors are collapsed into one vector per molecule.
   This is the step that makes the model work on graphs of different sizes: a mean over 30
   atoms and a mean over 70 atoms both give one vector of the same width.
4. **Head.** A small MLP maps that vector to the single number being predicted.

`FeRedoxGNN` below is those four stages and nothing else. Note the two details that keep
deep stacks trainable: `BatchNorm1d` after each convolution, and the residual connection
`h = h + h_in`, which lets a layer learn a correction rather than a replacement.


In [ ]:
class FeRedoxGNN(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, num_gnn_layers=3, dropout=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embed = nn.Linear(in_dim, hidden_dim)
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        for _ in range(num_gnn_layers):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
        self.dropout = nn.Dropout(dropout)
        # MLP head: two hidden layers
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.ReLU(),
            nn.Linear(hidden_dim//2, 1)
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        h = self.embed(x)
        for conv, bn in zip(self.convs, self.bns):
            h_in = h
            h = conv(h, edge_index)
            h = bn(h)
            h = F.relu(h)
            # simple residual connection
            h = h + h_in
            h = self.dropout(h)
        hg = global_mean_pool(h, batch)
        out = self.head(hg)
        return out.view(-1)

# Instantiate on the real feature width from 2.4. This model is here to be read, not
# trained; the model that gets trained is RexGCN in 3.2.
in_dim = demo_graphs[0].x.shape[1]
demo_model = FeRedoxGNN(in_dim=in_dim, hidden_dim=64, num_gnn_layers=3, dropout=0.1).to(device)
print(demo_model)
print('Trainable parameters:',
      sum(p.numel() for p in demo_model.parameters() if p.requires_grad))


### 3.2 RexGCN: the model this project uses

`FeRedoxGNN` calls `GCNConv` and lets PyG do the message passing. `RexGCN` writes its own
layer, because the standard GCN layer has a limitation that matters for these complexes:
**it ignores edge features.** A GCN message is built from the neighbor's node vector alone,
so a single bond and a dative metal-ligand bond look identical to it. For coordination
chemistry that is the wrong thing to throw away.

`RexGCNConv` is a `MessagePassing` subclass, which means PyG calls three methods in order:

| Method | Runs on | What this layer does |
|---|---|---|
| `message` | every edge | concatenates the neighbor's transformed features with that edge's 10 bond features, then scales by the GCN normalisation |
| `aggregate` | every node | sums the incoming messages (`aggr='add'`, set in the constructor) |
| `update` | every node | multiplies the summed message by `edge_updated` to fold the edge dimensions back down to `hidden_dim`, then adds the bias |

The normalisation is the $1/\sqrt{d_i d_j}$ factor from the GCN equation in 1.2, computed by
`gcn_norm`. It also adds self-loops, so a node keeps some of its own signal; the zero rows
appended to `edge_attr_prop` are the edge features of those self-loops, which by construction
carry no bond information.

`CFG.gnn.self_msg` chooses what happens to the node's own transformed vector: `'none'` drops
it, `'add'` adds it back (a residual connection), `'concat'` keeps both. The final model sets
it to `'add'`.

`RexGCN` then stacks those layers with `LayerNorm`, ReLU and dropout, pools with **both**
global max and global mean and concatenates the two, and runs the result through the fully
connected head. Concatenating two pooling functions is cheap and keeps information that
either one alone would destroy: the mean says what the molecule is like on average, the max
says whether any single atom is extreme.


In [ ]:
# The production model: a message-passing layer that uses edge features, and the network
# built from it. Class names and defaults are unchanged from the research notebook, except
# that the graph-level feature branch has been removed (see 3.3).

# Updated Final Model Version

import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Parameter, ModuleList, Linear, Identity, Dropout
from torch_geometric.nn import MessagePassing
from torch_geometric.nn import global_mean_pool as gap, global_max_pool as gmp
from torch_geometric.nn import GraphNorm, LayerNorm, BatchNorm
from torch_geometric.utils import add_remaining_self_loops, add_self_loops, remove_self_loops, softmax
from torch_geometric.nn.conv.gcn_conv import gcn_norm


# Helper functions for parameter initialization
def glorot(tensor):
    if tensor is not None:
        torch.nn.init.xavier_uniform_(tensor)

def zeros(tensor):
    if tensor is not None:
        torch.nn.init.zeros_(tensor)

# --- Configuration (Essential for RexGCNConv) ---
class CFG:
    class gnn:
        normalize_adj = True # Normalize or not
        self_msg = 'none'  # Options: 'none', 'add', 'concat'
        agg = 'add' # GCN uses 'add'
        msg_direction = 'single' # Not directly used by GCNConv but part of CFG

cfg = CFG()


# One message-passing layer. Inheriting from MessagePassing means PyG handles the
# gather/scatter over edges; this class only has to say what a message is (message),
# how messages combine (aggr='add'), and what to do with the result (update).
class RexGCNConv(MessagePassing):
    r""" General RexGCN Layer
    """
    def __init__(self, in_channels:int, out_channels:int, edge_dim:int, improved:bool=False, bias:bool=True,**kwards):
        super(RexGCNConv, self).__init__(aggr='add', **kwards)  # "Add" aggregation.
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.edge_dim = edge_dim
        self.improved = improved
        self.normalize = cfg.gnn.normalize_adj

        # Theta in the GCN equation: the linear map applied to node features.
        # edge_updated is the extra piece that folds the concatenated edge features
        # back down from (out_channels + edge_dim) to out_channels.
        self.weight = torch.nn.Parameter(torch.Tensor(in_channels, out_channels))
        self.edge_updated = torch.nn.Parameter(torch.Tensor(out_channels + edge_dim, out_channels))  # new property of GCN
        if bias:
            self.bias = torch.nn.Parameter(torch.Tensor(out_channels))
        else:
            self.register_parameter('bias', None)

        self.reset_parameters() # Calls glorot/zeros below

    def reset_parameters(self):
        glorot(self.weight) 
        glorot(self.edge_updated)
        zeros(self.bias)

    def forward(self, x: torch.Tensor, edge_index, edge_attr: torch.Tensor, edge_weight=None):
        '''
        x : node feature matrix that has shape [N, in_channels]
        edge_index : connectivity, Adj list in the edge index has shape [2, E]
        edge_attr: E-dimensional edge feature matrix that has shape [ E x edge_dim]
        '''
        # Linearly transform node feature matrix  (XΘ)
        x = torch.matmul(x, self.weight)

        edge_index_prop = edge_index # Start with original edge_index
        edge_attr_prop = edge_attr   # Start with original edge_attr
        norm = edge_weight           # Default norm is None or passed-in edge_weight

        # gcn_norm returns the 1/sqrt(d_i * d_j) coefficients and adds self-loops.
        # Self-loops need edge features too, so a block of zero rows is appended:
        # a self-loop is not a bond and must not look like one.
        if self.normalize:
            edge_index_norm, norm = gcn_norm(
                                            edge_index, 
                                            edge_weight, 
                                            x.size(self.node_dim),
                                            improved=self.improved,
                                            add_self_loops=True,
                                            dtype=x.dtype
                                        )
            edge_index_prop = edge_index_norm # Use normalized edge_index for propagation
            # Need edge_attr corresponding to edge_index_norm (i.e., with self-loop attrs)
            num_nodes = x.size(self.node_dim)
            self_loop_edges_attr = torch.zeros(num_nodes, edge_attr.size(1),
                                              dtype=edge_attr.dtype, device=edge_index.device)
            # Assuming add_remaining_self_loops appends loops at the end:
            edge_attr_prop = torch.cat([edge_attr, self_loop_edges_attr], dim=0)
        else:
            # No normalization case
            num_nodes = x.size(self.node_dim)
            self_loop_edges_attr = torch.zeros(num_nodes, edge_attr.size(1),
                                              dtype=edge_attr.dtype, device=edge_index.device)
            edge_attr_prop = torch.cat([edge_attr, self_loop_edges_attr], dim=0)

            edge_index_prop, _ = add_self_loops(edge_index, num_nodes=num_nodes)

            norm = edge_weight # norm remains None or edge_weight



        # Start propagating messages
        # Pass the prepared edge_index_prop, edge_attr_prop, and norm
        # Use x.size(self.node_dim) or equivalent for size
        x_msg =  self.propagate(edge_index_prop, x=x, edge_attr=edge_attr_prop, norm=norm) #, size=x.size(self.node_dim))

        # Handle self_msg based on config
        if cfg.gnn.self_msg == 'none':
            return x_msg
        elif cfg.gnn.self_msg == 'add':
            return x_msg + x # Add **transformed** input features (XΘ)
        elif cfg.gnn.self_msg == 'concat':
            return torch.cat([x_msg, x], dim=-1) # Concat with **transformed** input features
        else:
            raise ValueError('self_msg {} not defined'.format(
                cfg.gnn.self_msg))

    # Called once per edge. x_j is the source (neighbour) node's transformed features.
    def message(self, x_j, edge_attr, norm):
        # x_j: neighbor node features [E (+N), out_channels]
        # edge_attr: edge features [E (+N), edge_dim]
        if edge_attr is not None:
            # Ensure shapes match before cat
            # This assumes edge_attr corresponds row-wise to x_j after propagate setup
            msg = torch.cat([x_j, edge_attr], dim=-1)   # (E (+N)) x (out_channels + edge_dim)
        else:
            msg = x_j

        if norm is not None:
            # norm should have shape [E (+N)]
            return norm.view(-1, 1) * msg
        else:
            return msg

    # Called once per node, after the incoming messages have been summed.
    def update(self, aggr_out):   # Return node embeddings
        # aggr_out: [N, out_channels + edge_dim] if edge_attr included in message, else [N, out_channels]
        # Based on message(), it includes edge_attr, so shape is [N, out_channels + edge_dim]
        aggr_out = torch.mm(aggr_out, self.edge_updated) # [N, out_channels]
        if self.bias is not None:
            return aggr_out + self.bias
        else:
            return aggr_out

    def __repr__(self):
        return '{}({}, {}, {})'.format(self.__class__.__name__, self.in_channels, self.out_channels, self.edge_dim)



# The full network: embed through conv layers, pool, then a fully connected head.
class RexGCN(torch.nn.Module):
    def __init__(self, node_features, hidden_dim, edge_features, dropout,
                 num_conv_layers, num_fc_layers, norm_type='layer'):
        super(RexGCN, self).__init__()
        
        self.node_features = node_features
        self.hidden_dim = hidden_dim
        self.edge_features = edge_features
        self.dropout_rate = dropout
        self.num_conv_layers = num_conv_layers
        self.num_fc_layers = num_fc_layers
        self.norm_type = norm_type.lower() if isinstance(norm_type, str) else 'none'

        self.conv_list = ModuleList()
        self.norm_list = ModuleList()

        # --- GNN Layers ---
        current_dim = node_features
        for i in range(num_conv_layers):
            self.conv_list.append(
                RexGCNConv(current_dim, hidden_dim, edge_features) 
            )

            if self.norm_type == 'layer':
                self.norm_list.append(LayerNorm(hidden_dim))
            elif self.norm_type == 'graph':
                self.norm_list.append(GraphNorm(hidden_dim)) 
            elif self.norm_type == 'batch':
                 self.norm_list.append(BatchNorm(hidden_dim))
            elif self.norm_type is None or self.norm_type.lower() == 'none':
                 self.norm_list.append(Identity()) 
            else:
                raise ValueError(f"Unsupported norm_type: {self.norm_type}")

            current_dim = hidden_dim 

        # --- FC Layers ---
        self.fc_list = ModuleList()
        
        # CALCULATE INPUT DIMENSION:
        # Graph part only: hidden_dim * 2, because the pooled vector is concat(gmp, gap)
        fc_input_dim = hidden_dim * 2
        
        current_fc_dim = fc_input_dim
        
        for i in range(num_fc_layers - 1):
            self.fc_list.append(Linear(current_fc_dim, hidden_dim*2))
            current_fc_dim = hidden_dim*2

        self.fc_out = Linear(current_fc_dim, 1)
        self.dropout = Dropout(p=self.dropout_rate)


    def forward(self, data):
        x, edge_index, batch_index, edge_attr = data.x, data.edge_index, data.batch, data.edge_attr

        # Run GNN
        for i in range(self.num_conv_layers):
            x = self.conv_list[i](x, edge_index, edge_attr)
            x = F.relu(x)
            x = self.dropout(x)
            if i < len(self.norm_list):
                 norm = self.norm_list[i]
                 if self.norm_type in ['graph', 'layer']:
                     x = norm(x, batch_index)
                 else:
                     x = norm(x)

        # Readout: max-pool and mean-pool the node vectors, then concatenate. This is
        # the step that turns a variable number of atoms into one fixed-width vector.
        # Pool Graph Embeddings
        x_pooled = torch.cat([gmp(x, batch_index), gap(x, batch_index)], dim=1)

        # No graph-level features are concatenated here (see 3.3 and Exercise 1)
        x = x_pooled

        # FC Layers
        for i in range(self.num_fc_layers - 1):
            x = self.fc_list[i](x)
            x = F.relu(x)
            x = self.dropout(x)

        x = self.fc_out(x)
        return x


### 3.3 What this model does not use

Look at `forward` again. It reads `data.x`, `data.edge_index`, `data.edge_attr` and
`data.batch`, and nothing else. **Node features and edge features only.**

Meanwhile every `Data` object built in Part 2 carries `morgan_fp`, `metal_charge`,
`metal_neigh_symbols`, `metal_neigh_charge`, `z` and `pos`. The pipeline computes them, the
batches carry them onto the GPU, and the model steps straight past them. The graph-level
level of Day 3 section 6.3 is present in the data and absent from the model.

That is a deliberate choice for this notebook, not an oversight in the research code: the
original model concatenates a fingerprint onto the pooled vector before the head. It is
also the most natural thing to change first, which is why it is Exercise 1.


## Part 4 - Training machinery

Three pieces are needed before anything can be trained: a learning-rate schedule, the loss
and the loops that use it, and a place to keep the hyperparameters.

### 4.1 The learning-rate schedule

A fixed learning rate is a compromise: large enough to make progress early, small enough not
to thrash later, and therefore wrong at both ends. `NoamLR` splits the difference in time
instead. It ramps linearly from `init_lr` to `max_lr` over the first `warmup_epochs`, then
decays exponentially to `final_lr` over the remaining epochs. The warm-up matters most at the
start, when the randomly initialised weights would otherwise take a large step in a direction
that means nothing.


In [ ]:
# Learning-rate schedule: linear warm-up to max_lr, then exponential decay to final_lr.
# Note that step() is called once per batch, not once per epoch.

from torch.optim.lr_scheduler import _LRScheduler
from typing import Dict, Iterator, List, Optional, Union, OrderedDict, Tuple

class NoamLR(_LRScheduler):
    """
    Noam learning rate scheduler with piecewise linear increase and exponential decay.

    The learning rate increases linearly from init_lr to max_lr over the course of
    the first warmup_steps (where :code:`warmup_steps = warmup_epochs * steps_per_epoch`).
    Then the learning rate decreases exponentially from :code:`max_lr` to :code:`final_lr` over the
    course of the remaining :code:`total_steps - warmup_steps` (where :code:`total_steps =
    total_epochs * steps_per_epoch`). This is roughly based on the learning rate
    schedule from `Attention is All You Need <https://arxiv.org/abs/1706.03762>`_, section 5.3.
    """
    def __init__(self,
                 optimizer: 'Optimizer',
                 warmup_epochs: List[Union[float, int]],
                 total_epochs: List[int],
                 steps_per_epoch: int,
                 init_lr: List[float],
                 max_lr: List[float],
                 final_lr: List[float]):

        assert len(optimizer.param_groups) == len(warmup_epochs) == len(total_epochs) == len(init_lr) == \
               len(max_lr) == len(final_lr)

        self.num_lrs = len(optimizer.param_groups)

        self.optimizer = optimizer
        self.warmup_epochs = np.array(warmup_epochs)
        self.total_epochs = np.array(total_epochs)
        self.steps_per_epoch = steps_per_epoch
        self.init_lr = np.array(init_lr)
        self.max_lr = np.array(max_lr)
        self.final_lr = np.array(final_lr)

        self.current_step = 0
        self.lr = init_lr
        self.warmup_steps = (self.warmup_epochs * self.steps_per_epoch).astype(int)
        self.total_steps = self.total_epochs * self.steps_per_epoch
        self.linear_increment = (self.max_lr - self.init_lr) / self.warmup_steps

        self.exponential_gamma = (self.final_lr / self.max_lr) ** (1 / (self.total_steps - self.warmup_steps))

        super(NoamLR, self).__init__(optimizer)

    def get_lr(self) -> List[float]:
        return list(self.lr)

    def step(self, current_step: int = None):
        if current_step is not None:
            self.current_step = current_step
        else:
            self.current_step += 1

        for i in range(self.num_lrs):
            if self.current_step <= self.warmup_steps[i]:
                self.lr[i] = self.init_lr[i] + self.current_step * self.linear_increment[i]
            elif self.current_step <= self.total_steps[i]:
                self.lr[i] = self.max_lr[i] * (self.exponential_gamma[i] ** (self.current_step - self.warmup_steps[i]))
            else:  # theoretically this case should never be reached since training should stop at total_steps
                self.lr[i] = self.final_lr[i]

            self.optimizer.param_groups[i]['lr'] = self.lr[i]




### 4.2 Loss and the train/evaluate loops

Training minimises MSE, because it is smooth and cheap to differentiate. Reporting uses RMSE,
because it is in volts and can be compared directly with the Day 3 baselines. `RmseLoss`
exists so RMSE can also be used as a loss when wanted; the $\epsilon$ inside the square root
keeps the gradient finite when the error approaches zero.

Both loops accumulate `loss * data.num_graphs` and divide by the dataset size at the end,
which weights the last, possibly smaller, batch correctly. `evaluate` runs under
`torch.no_grad()`: no gradients are needed, and switching them off saves memory.


In [ ]:
# Loss function and the two loops used everywhere below: one pass over the training set with
# gradients, one pass over any loader without them.

# Training functions

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

class RmseLoss(torch.nn.Module):
    def __init__(self, eps=1e-6):
        super().__init__()
        # Instantiate the MSELoss module. It defaults to reduction='mean'
        self.mse = torch.nn.MSELoss()
        self.eps = eps # Epsilon for numerical stability near sqrt(0)

    def forward(self, y_pred, y_true):
        # Ensure y_true has the same shape as y_pred for MSE calculation
        y_true = y_true.view_as(y_pred)

        # Calculate MSE loss (averaged over the batch by default)
        loss_mse = self.mse(y_pred, y_true)
        # Calculate RMSE
        loss_rmse = torch.sqrt(loss_mse + self.eps)

        return loss_rmse
    

rmse_loss = RmseLoss() # Instantiate custom loss

def train(train_loader, model, optimizer, scheduler, loss_type='mse'):
    model.train()
    train_loss=0
    for data in train_loader:
        data = data.to(device)
        model.zero_grad()
        y_pred = model(data) #(data.x, data.edge_index, data.batch,data.edge_attr)
        if loss_type == 'mse':
            loss_train = F.mse_loss(y_pred, data.y)   # rmse_loss(y_pred, data.y) #
            loss_train.backward()
        elif loss_type == 'rmse':
            loss_train = rmse_loss(y_pred, data.y)
            loss_train.backward()
        optimizer.step()
        if isinstance(scheduler, NoamLR):
            scheduler.step()
        train_loss += float(loss_train) * data.num_graphs
    return train_loss / len(train_loader.dataset)

def train_rmse(train_loader,model, optimizer, scheduler,):
    model.train()
    train_loss=0
    for data in train_loader:
        data = data.to(device)
        model.zero_grad()
        y_pred = model(data) #(data.x, data.edge_index, data.batch,data.edge_attr)
        loss_train = rmse_loss(y_pred, data.y) #
        loss_train.backward()
        optimizer.step()
        if isinstance(scheduler, NoamLR):
            scheduler.step()
        train_loss += float(loss_train) * data.num_graphs
    return train_loss / len(train_loader.dataset)


def evaluate(loader, model):
    model.eval()
    test_loss=0
    for data_t in loader:
        data_t = data_t.to(device)
        with torch.no_grad():
            out = model(data_t) #(data_t.x, data_t.edge_index, data_t.batch,data_t.edge_attr)
            loss_test = F.mse_loss(out, data_t.y)
        test_loss += float(loss_test) * data_t.num_graphs
    return test_loss / len(loader.dataset)


def evaluate_rmse(loader, model):
    model.eval()
    test_loss=0
    for data_t in loader:
        data_t = data_t.to(device)
        with torch.no_grad():
            out = model(data_t) #(data_t.x, data_t.edge_index, data_t.batch,data_t.edge_attr)
            loss_test = rmse_loss(out, data_t.y)
        test_loss += float(loss_test) * data_t.num_graphs
    return test_loss / len(loader.dataset)

### 4.3 Hyperparameters

Everything that could be tuned, collected in one class so a run is described by a single
object. These are the values used for the published model.


In [ ]:
# Every hyperparameter in one place. Exercise 2 asks you to change some of these and rerun
# the cross-validation below.

# --- Model Setup ---

class TrainArgsGCN:
    def __init__(self,
                 edge_features=None,  # Provide default values
                 num_features=None,   # Provide default values
                 dropout=0.1,
                 norm_type='layer',
                 num_conv_layers=3,
                 num_fc_layers=5,
                 hidden_dim=512,
                 extra_dim = 64,
                 batch_size=128,
                 init_lr=1e-4,
                 max_lr=1e-3,
                 final_lr=1e-4,
                 num_lrs=1,
                 warmup_epochs=2.0,
                 epochs=200,
                 patience=50):
        # Assign arguments to instance attributes
        self.edge_features = edge_features
        self.num_features = num_features
        self.dropout = dropout
        self.norm_type = norm_type
        self.num_conv_layers = num_conv_layers
        self.num_fc_layers = num_fc_layers
        self.hidden_dim = hidden_dim
        self.extra_dim = extra_dim
        self.batch_size = batch_size
        self.init_lr = init_lr
        self.max_lr = max_lr
        self.final_lr = final_lr
        self.num_lrs = num_lrs
        self.warmup_epochs = warmup_epochs
        self.epochs = epochs
        self.patience = patience


device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

## Part 5 - Cross-validation

### 5.1 Why a single split is not an answer

Day 2 section 8.2 introduced the train/validation/test split, and Day 3 trained the baselines
on exactly one such split. There is a problem with reporting the number that comes out of it.

Which complexes land in the validation set is a matter of luck. Change `random_state` and the
split changes, and with 1546 complexes a validation fold holds only a few hundred; a handful
of unusual complexes falling on one side rather than the other moves the RMSE noticeably. So
a single number like "validation RMSE = 0.31 V" is one draw from a distribution, and on its
own it does not say whether 0.31 is what the model does or what this split does.

**k-fold cross-validation** measures the distribution instead. Split the data into $k$ equal
parts, then train $k$ times: each run holds out a different part for validation and trains on
the other $k-1$. Every complex is validated exactly once, and the result is $k$ scores whose
**mean and standard deviation** are the honest report:

$$
\mathrm{RMSE}_{\mathrm{CV}} = \frac{1}{k}\sum_{i=1}^{k} \mathrm{RMSE}_i
\qquad
s = \sqrt{\frac{1}{k}\sum_{i=1}^{k}\left(\mathrm{RMSE}_i - \mathrm{RMSE}_{\mathrm{CV}}\right)^2}
$$

Two reasons this matters more than it might seem:

- **It tells you when a difference is real.** If architecture A scores 0.30 and B scores 0.28
  on one split, that looks like a win for B. If the fold-to-fold spread is $\pm 0.04$, it is
  not a win, it is noise. Without CV there is no way to tell, and it is the most common way
  model comparisons go wrong.
- **It is a measure of the procedure, not of one model.** Each fold trains a different model
  and they are all thrown away. What survives is an estimate of how well *this recipe*
  performs on data it has not seen, which is exactly the quantity you need before committing
  to a final model trained on everything, in the sense of Day 2 section 8.4.

The cost is that everything takes $k$ times as long. With $k = 5$ and 200 epochs per fold,
this is a GPU job, not something to run while someone is watching.

The split here is **stratified**: complexes are labelled by the sign of their redox potential
and each fold is built to hold the same proportion of each. Without that, a fold could end up
short of negative-potential complexes purely by chance, which would add variance to the very
number the exercise is trying to measure.

### 5.2 The k-fold run

Note that the graphs are rebuilt inside the loop, from the fold's own dataframe, rather than
built once outside it. That is deliberate: anything fitted on data must be fitted on the
training part of that fold only, and rebuilding keeps the boundary obvious. Day 2 section 8.3
called this leakage.


In [ ]:
# 5-fold stratified cross-validation. Each fold: rebuild the graphs, re-initialise the model
# from scratch, train for args.epochs, and keep the best validation RMSE reached.
# This is the long cell in the notebook. On a GPU it is minutes per fold; on a CPU, do not.

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error
import copy
import matplotlib.pyplot as plt
import torch.optim as optim
from collections import Counter
import math # For loss plot adjustment
import copy

print("--- Preparing for K-Fold Cross-Validation on Entire Dataset ---")
labels_for_stratification = []
for data in final_df.y:
    y_value = data
    if y_value < 0:
        labels_for_stratification.append(-1)
    else:
        labels_for_stratification.append(1)

print(f"Extracted {len(labels_for_stratification)} labels for stratification.")
print("Original Label Distribution:", Counter(labels_for_stratification))

indices = np.array(list(range(len(final_df)))) # Use NumPy array for easier indexing with KFold output
labels_for_stratification = np.array(labels_for_stratification) # Also NumPy array

# --- K-Fold Cross-Validation Setup ---

k_folds = 5 # Number of folds (e.g., 5 or 10)
skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)

# Lists to store metrics from each fold
fold_best_val_rmses = []
fold_best_val_mses = [] # Store best validation MSE for each fold
all_train_losses_cv = [] # Store training losses for each fold
all_train_losses_cv_rmse = [] # Store training RMSE
all_val_losses_cv = []   # Store validation losses for each fold
all_val_losses_cv_rmse = []   # Store validation losses for each fold


print(f'\n--- Starting {k_folds}-Fold Cross-Validation ---')

# --- K-Fold Loop ---
# Use the full indices and labels for splitting
for fold, (train_fold_idx, val_fold_idx) in enumerate(skf.split(indices, labels_for_stratification)):
    print(f'\n===== Fold {fold+1}/{k_folds} =====')

    # --- Get Original Indices for this Fold (Directly from KFold output) ---
    original_train_fold_indices = indices[train_fold_idx]
    original_val_fold_indices = indices[val_fold_idx]

    # --- Create Data for this Fold ---

    # get train df
    train_fold_df = final_df.iloc[original_train_fold_indices]
    
    # get val df
    val_fold_df = final_df.iloc[original_val_fold_indices]
    
    # Generate graph data
    train_fold_data = GenGraphs(train_fold_df) #[graph_data[i] for i in original_train_fold_indices]
    val_fold_data = GenGraphs(val_fold_df) #[graph_data[i] for i in original_val_fold_indices]
    
    num_node_features = train_fold_data[0].x.size(1)
    num_edge_features = train_fold_data[0].edge_attr.size(1)
    print(f"Number of node features: {num_node_features}")
    print(f"Number of edge features: {num_edge_features}")
    print(f'Fold {fold+1} - Train size: {len(train_fold_data)}, Validation size: {len(val_fold_data)}')

    # train args
    args = TrainArgsGCN(edge_features = num_edge_features, num_features = num_node_features)

    # --- Create DataLoaders for this Fold ---
    train_fold_loader = DataLoader(train_fold_data, batch_size=args.batch_size, shuffle=True)
    val_fold_loader = DataLoader(val_fold_data, batch_size=args.batch_size, shuffle=False)

    # A fresh model every fold. Carrying weights over from the previous fold would mean
    # this fold's validation complexes had already been trained on.
    # --- Re-initialize Model, Optimizer, and Scheduler for each fold ---
    print(f'Fold {fold+1} - Initializing model, optimizer, scheduler...')
    model = RexGCN(node_features=args.num_features,
                   hidden_dim=args.hidden_dim,
                   edge_features=args.edge_features,
                   norm_type=args.norm_type,
                   dropout=args.dropout,
                   num_fc_layers=args.num_fc_layers,
                   num_conv_layers=args.num_conv_layers
                   ).to(device)

    params = [{'params': model.parameters(), 'lr': args.init_lr, 'weight_decay': 0}]
    optimizer = optim.Adam(params)
    criterion = torch.nn.MSELoss()
    rmse_loss = RmseLoss()

    scheduler = NoamLR(
        optimizer=optimizer,
        warmup_epochs=[args.warmup_epochs],
        total_epochs=[args.epochs] * args.num_lrs,
        steps_per_epoch=len(train_fold_loader) // args.batch_size,
        init_lr=[args.init_lr],
        max_lr=[args.max_lr],
        final_lr=[args.final_lr]
    )

    # --- Training Loop for this Fold ---
    patience = args.patience
    best_val_loss_fold = float('inf')
    patience_counter_fold = 0
    best_model_state_fold = None
    best_epoch_fold = -1

    train_losses_fold = []
    train_losses_rmse_fold = []
    val_losses_fold = [] # Store MSE
    val_losses_rmse_fold = [] # Store RMSE

    print(f'Fold {fold+1} - Starting training...')
    for epoch in range(1, args.epochs + 1):
        # Use MSE for training gradients, RMSE for validation metric
        train_loss = train(train_fold_loader, model=model, optimizer=optimizer, scheduler=scheduler, loss_type='mse') 

        # Calculate Train RMSE from MSE (Faster than running evaluate_rmse)
        train_loss_rmse = math.sqrt(train_loss)

        val_loss_mse = evaluate(val_fold_loader, model=model) 
        # Calculate Val RMSE from MSE (Faster than running evaluate_rmse)
        #val_loss_rmse = evaluate_rmse(val_fold_loader, model=model)
        val_loss_rmse = math.sqrt(val_loss_mse)
        current_val_loss = val_loss_rmse # Use RMSE for early stopping

        train_losses_fold.append(train_loss)
        train_losses_rmse_fold.append(train_loss_rmse)
        val_losses_fold.append(val_loss_mse)
        val_losses_rmse_fold.append(val_loss_rmse)

        if epoch % 10 == 0 or epoch == 1: # Print less frequently
            print(f'  Epoch: {epoch:d}, Loss - Train: {train_loss:.5f}, Val MSE: {val_loss_mse:.5f}, Val RMSE: {val_loss_rmse:.5f}')

        # --- Early Stopping Check for this Fold ---
        improved = False
        if current_val_loss < best_val_loss_fold:
            best_val_loss_fold = current_val_loss
            improved = True

        if improved:
            patience_counter_fold = 0
            best_model_state_fold = copy.deepcopy(model.state_dict())
            best_epoch_fold = epoch
            # print(f'    -> Fold {fold+1} Val loss improved to {best_val_loss_fold:.5f}. Saving state.')
        else:
            patience_counter_fold += 1
            # print(f'    -> Fold {fold+1} Val metric did not improve. Patience: {patience_counter_fold}/{patience}')

        #if patience_counter_fold >= patience:
        #    print(f'  !!! Fold {fold+1} Early stopping triggered after epoch {epoch}. Best epoch was {best_epoch_fold} with Val RMSE: {best_val_loss_fold:.5f}')
        #    break

    # --- Store performance for this fold ---
    # Store the best validation RMSE achieved during this fold's training
    if best_epoch_fold != -1: # Check if training improved at all
         fold_best_val_rmses.append(best_val_loss_fold)
         fold_best_val_mses.append(val_losses_fold[best_epoch_fold - 1]) # Store MSE at best epoch
         print(f'Fold {fold+1} - Finished. Best Validation RMSE: {best_val_loss_fold:.5f} (at Epoch {best_epoch_fold})')
         print(f'Fold {fold+1} - Best Validation MSE: {val_losses_fold[best_epoch_fold - 1]:.5f} (at Epoch {best_epoch_fold})')
    else:
         print(f'Fold {fold+1} - Finished. Validation loss did not improve.')
         # Decide how to handle this: append NaN, append last loss, or skip? NaN is often best for averaging.
         fold_best_val_rmses.append(float('nan'))

    # Store losses for plotting average curves
    all_train_losses_cv.append(train_losses_fold)
    all_train_losses_cv_rmse.append(train_losses_rmse_fold)
    all_val_losses_cv.append(val_losses_fold)
    all_val_losses_cv_rmse.append(val_losses_rmse_fold)



# --- Aggregate and Report Results ---
print('\n--- Cross-Validation Summary ---')

# Calculate average and std dev of the *best validation RMSE* achieved in each fold
avg_val_rmse = np.nanmean(fold_best_val_rmses)
std_val_rmse = np.nanstd(fold_best_val_rmses)

avg_val_mse = np.nanmean(fold_best_val_mses)
std_val_mse = np.nanstd(fold_best_val_mses)

print(f'Average Best Validation RMSE across {k_folds} folds: {avg_val_rmse:.5f} ± {std_val_rmse:.5f}')
print(f'Average Best Validation MSE across {k_folds} folds: {avg_val_mse:.5f} ± {std_val_mse:.5f}')

print('\nIndividual Fold Best Validation RMSEs (MSEs):')
for i, rmse in enumerate(fold_best_val_rmses):
    print(f'  Fold {i+1}: {rmse:.5f} (MSE: {fold_best_val_mses[i]:.5f})')


### 5.3 Average loss curves

The mean training and validation curve across the folds, with the fold-to-fold spread as a
shaded band. The band is the point of the figure: it is the uncertainty on every number
quoted from a single split.


In [ ]:
# Pad the per-fold curves to equal length, average them, and plot mean +/- one standard
# deviation. The tab-separated dump alongside the figure is for replotting later.

# --- Plot Average Loss Curves ---

max_epochs_run = max(len(losses) for losses in all_val_losses_cv) if all_val_losses_cv else args.epochs

padded_train_losses = [l + [np.nan]*(max_epochs_run - len(l)) for l in all_train_losses_cv]
padded_train_losses_rmse = [l + [np.nan]*(max_epochs_run - len(l)) for l in all_train_losses_cv_rmse]
padded_val_losses = [l + [np.nan]*(max_epochs_run - len(l)) for l in all_val_losses_cv]
padded_val_losses_rmse = [l + [np.nan]*(max_epochs_run - len(l)) for l in all_val_losses_cv_rmse] #all_val_losses_cv]

avg_train_loss = np.nanmean(padded_train_losses, axis=0)
std_train_loss = np.nanstd(padded_train_losses, axis=0)
avg_train_loss_rmse = np.nanmean(padded_train_losses_rmse, axis=0)
std_train_loss_rmse = np.nanstd(padded_train_losses_rmse, axis=0)

avg_val_loss = np.nanmean(padded_val_losses, axis=0)
std_val_loss = np.nanstd(padded_val_losses, axis=0)
avg_val_loss_rmse = np.nanmean(padded_val_losses_rmse, axis=0)
std_val_loss_rmse = np.nanstd(padded_val_losses_rmse, axis=0)

epochs_axis = range(1, max_epochs_run + 1)



# Saving plot data
plot_data_df = pd.DataFrame({
    'epoch': epochs_axis,
    'train_mse_avg': avg_train_loss,
    'train_mse_std': std_train_loss,
    'train_rmse_avg': avg_train_loss_rmse,
    'train_rmse_std': std_train_loss_rmse,
    'val_mse_avg': avg_val_loss,
    'val_mse_std': std_val_loss,
    'val_rmse_avg': avg_val_loss_rmse,
    'val_rmse_std': std_val_loss_rmse
})

# Save as tab-separated values
txt_save_path = FIG_DIR / 'GCN_kfold-cv_data.txt'
plot_data_df.to_csv(txt_save_path, sep='\t', index=False)
print(f"Plot data saved to {txt_save_path}")



# Plotting 

train_plot = avg_train_loss_rmse
train_fill = std_train_loss_rmse

val_plot = avg_val_loss_rmse
val_fill = std_val_loss_rmse

fig, ax = plt.subplots(figsize=(12, 8), dpi=100)

# Publication Ready Styling
plt.rcParams.update({
    'font.size': 25,
    'font.family': 'sans-serif',
    
    # Axes
    'axes.linewidth': 2,
    'axes.labelsize': 35,
    'axes.titlesize': 35,

    # ticks
    'xtick.labelsize': 30,
    'ytick.labelsize': 30,
    'xtick.major.width': 2,
    'xtick.major.size': 8,
    'xtick.minor.width': 1.5,
    'xtick.minor.size': 5,
    'ytick.major.width': 2,
    'ytick.major.size': 8,
    'ytick.minor.width': 1.5,
    'ytick.minor.size': 5,

    # legend
    'legend.fontsize': 30,
})

plt.plot(epochs_axis, train_plot, label='Train', color='blue')
plt.fill_between(epochs_axis, train_plot - train_fill, train_plot + train_fill, color='blue', alpha=0.2)

plt.plot(epochs_axis, val_plot, label=f'Validation', color='orange')
plt.fill_between(epochs_axis, val_plot - val_fill, val_plot + val_fill, color='orange', alpha=0.2)

#plt.title(f'Average Training & Validation Loss ({k_folds}-Fold CV)', fontsize=25)
ax.set_title("RexGCN", pad=10)
plt.xlabel('Epochs') #, fontsize=35)
plt.ylabel('RMSE Loss (V)') #, fontsize=35)


#plt.grid(True)

plt.xlim(0, max_epochs_run)
plt.ylim(0, 0.8)
#min_loss_val = np.nanmin(avg_val_loss - std_val_loss) if not np.all(np.isnan(avg_val_loss)) else 0
#max_loss_val = np.nanmax(avg_val_loss + std_val_loss) if not np.all(np.isnan(avg_val_loss)) else 1
#min_loss_train = np.nanmin(avg_train_loss - std_train_loss) if not np.all(np.isnan(avg_train_loss)) else 0
#max_loss_train = np.nanmax(avg_train_loss + std_train_loss) if not np.all(np.isnan(avg_train_loss)) else 1

# Tick Formatting
# Fixed number of major ticks to keep it clean
ax.xaxis.set_major_locator(MaxNLocator(nbins=6))
ax.yaxis.set_major_locator(MaxNLocator(nbins=5))

# Minor ticks (AutoMinorLocator(4) creates 3 minor ticks between majors)
ax.xaxis.set_minor_locator(AutoMinorLocator(4))
ax.yaxis.set_minor_locator(AutoMinorLocator(4))

ax.tick_params(direction='in', top=True, right=True, which='both')

# Legend
legend = ax.legend(loc='upper left', frameon=True, labelcolor='linecolor') #, fontsize=25)
frame = legend.get_frame()
frame.set_alpha(0.5)
frame.set_edgecolor('black')
frame.set_linewidth(1.0)

# increase font size of ticks
#plt.tick_params(axis='both', which='major', labelsize=30)

plt.savefig(FIG_DIR / 'GCN_kfold-cv.png', transparent=True, bbox_inches='tight')


**Mentor checkpoint 10**: after cross-validation

- What is the mean validation RMSE and the spread across folds? How does the mean compare
  with the Random Forest and GPR numbers from Day 3 Part 1?
- Is the gap between the GNN and the baselines larger than the fold-to-fold spread?
- Do the average train and validation curves separate? What would it mean if they did?
- Why is a new model instantiated inside the fold loop rather than outside it?

Proceed only after confirmation.

---


## Part 6 - The final model

Cross-validation measured the recipe. Now train one model to keep, on a fixed
70 / 15 / 15 stratified split, and evaluate it on the test portion.

### 6.1 Splitting the data


In [ ]:
# A single stratified 70/15/15 split, then graphs for each part. Stratification is on the
# sign of the redox potential, the same labelling used by the cross-validation above.

# Data splitting

from torch_geometric.loader import DataLoader
import random
from sklearn.model_selection import train_test_split

labels_for_stratification = [] #[data.y.item() for data in graph_data]
for data in final_df.y:
    y_value = data
    # Assign label based on the sign of the value
    if y_value < 0:
        labels_for_stratification.append(-1) # Assign -1 for negative values
    else: # Includes y_value >= 0 (including exactly 0)
        labels_for_stratification.append(1)  # Assign +1 for non-negative values

print(f"Extracted {len(labels_for_stratification)} labels for stratification.")
# Optional: Check label distribution
from collections import Counter
print("Original Label Distribution:", Counter(labels_for_stratification))


train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15 # train + val + test = 1.0

# Indices for splitting data and labels together
indices = list(range(len(final_df)))

# First split: Separate Test set, stratifying by the labels
train_val_indices, test_indices, y_train_val, y_test_unused = train_test_split(
    indices, # Split indices first
    labels_for_stratification, # Labels corresponding to indices
    test_size=test_ratio,
    shuffle=True, # Shuffle before splitting
    stratify=labels_for_stratification, # Stratify based on the full label list
    random_state=42
)

# Second split: Separate Train and Validation sets from the train_val pool
# Stratify based on the labels remaining in the train_val pool (y_train_val)
# Calculate the validation proportion relative to the train_val pool size
val_proportion_of_train_val = val_ratio / (train_ratio + val_ratio)

train_indices, val_indices, y_train_unused, y_val_unused = train_test_split(
    train_val_indices, # Split the indices reserved for train/val
    y_train_val,       # Labels corresponding to train_val_indices
    test_size=val_proportion_of_train_val,
    shuffle=True, # shuffle False does not work with stratification lol
    stratify=y_train_val, # Stratify based on the train_val labels
    random_state=42 # Can use same state or different
)

# Create the final data splits using the selected indices

# get train df
train_fold_df = final_df.iloc[train_indices]

# get val and test df
val_fold_df = final_df.iloc[val_indices]
test_fold_df = final_df.iloc[test_indices]

# Generate graph data
train_data = GenGraphs(train_fold_df)  # [graph_data[i] for i in train_indices]
val_data = GenGraphs(val_fold_df)   # [graph_data[i] for i in val_indices]
test_data = GenGraphs(test_fold_df)   # [graph_data[i] for i in test_indices]

num_node_features = train_data[0].x.size(1)
num_edge_features = train_data[0].edge_attr.size(1)

# --- Verification ---
print(f'\n--- Split Sizes ---')
print(f'Train size: {len(train_data)}')
print(f'Validation size: {len(val_data)}')
print(f'Test size: {len(test_data)}')
print(f'Total: {len(train_data) + len(val_data) + len(test_data)} (should match original)')

print(f'\n--- Class Distribution Verification ---')
train_labels_orig = [d.y.item() for d in train_data]
val_labels_orig = [d.y.item() for d in val_data]
test_labels_orig = [d.y.item() for d in test_data]

print(f"Train +/- ratio: {sum(1 for y in train_labels_orig if y>=0)} / {sum(1 for y in train_labels_orig if y<0)}")
print(f"Val +/- ratio: {sum(1 for y in val_labels_orig if y>=0)} / {sum(1 for y in val_labels_orig if y<0)}")
print(f"Test +/- ratio: {sum(1 for y in test_labels_orig if y>=0)} / {sum(1 for y in test_labels_orig if y<0)}")


### 6.2 Instantiating the model

`cfg.gnn.self_msg` is switched from `'none'` to `'add'` here, so each layer adds the node's
own transformed vector back to the aggregated message. The parameter count printed at the end
is worth noting before comparing with the architectures in Part 7.


In [ ]:
# Build the model that will actually be kept. TrainArgsGCN is already defined in 4.3, so only
# the instance and the model are created here.

# --- Initialization Setup ---


args = TrainArgsGCN(edge_features = num_edge_features, num_features = num_node_features)

#device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# Adjust these based on how you want RexGCNConv to behave internally
cfg.gnn.normalize_adj = True  
cfg.gnn.self_msg = 'add'     # Use internal skip connection? 'none', 'add', 'concat'

# --- Instantiate the RexGCN model ---
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

model_gcn = RexGCN(node_features=args.num_features,
                   hidden_dim=args.hidden_dim,
                   edge_features=args.edge_features,
                   norm_type = args.norm_type,
                   dropout=args.dropout,
                   num_fc_layers=args.num_fc_layers,
                   num_conv_layers=args.num_conv_layers
                   ).to(device)

# --- Print Model Info ---
print(model_gcn)
print("Number of parameters (RexGCN): ", sum(p.numel() for p in model_gcn.parameters() if p.requires_grad))



### 6.3 Training with early stopping

The validation RMSE is checked every epoch. Whenever it improves, a copy of the weights is
kept; when it fails to improve for `patience` epochs in a row, training stops and the best
copy is restored. The final numbers therefore come from the best epoch, not the last one.


In [ ]:
# The training loop. Early stopping keeps a deepcopy of the best weights seen, so the model
# evaluated at the end is the best one, not whatever the last epoch produced.

# define the optimizer and 
import copy

model = model_gcn
args = args

# optimizer
params = [{'params': model.parameters(), 'lr': args.init_lr, 'weight_decay': 0}]
optimizer = torch.optim.Adam(params)
criterion = torch.nn.MSELoss()  # Example for regression


# DataLoaders
batch_size = args.batch_size
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

train_loader_noshuffle = DataLoader(train_data, batch_size=batch_size, shuffle=False)

print("\nDataLoaders created successfully.")

#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

scheduler = NoamLR(
    optimizer=optimizer,
    warmup_epochs=[args.warmup_epochs],
    total_epochs=[args.epochs] * args.num_lrs,
    steps_per_epoch=len(train_loader) // args.batch_size,
    init_lr=[args.init_lr],
    max_lr=[args.max_lr],
    final_lr=[args.final_lr]
)


# Early Stopping variables
patience = args.patience # Number of epochs to wait for improvement before stopping
best_val_loss = float('inf')
patience_counter = 0
best_model_state = None

#train  
train_losses = []
train_losses_rmse = []
val_losses = []
val_losses_rmse = []
test_losses = []
for epoch in range(1, args.epochs):
    #train_loss = train(train_loader) #train(train_loader)
    # train with mse loss
    train_loss = train(train_loader, model=model, optimizer=optimizer, scheduler=scheduler, loss_type='mse') #train(train_loader)
    # Calculate Train RMSE from MSE (Faster than running evaluate_rmse)
    train_loss_rmse = math.sqrt(train_loss)
    
    val_loss = evaluate(val_loader, model=model) #test(test_loader)
    val_loss_rmse = evaluate_rmse(val_loader, model=model) #test(test_loader)
    current_val_loss = val_loss_rmse

    train_losses.append(train_loss)
    train_losses_rmse.append(train_loss_rmse)
    val_losses.append(val_loss)
    val_losses_rmse.append(val_loss_rmse)
    if epoch % 5 == 0:
        print(f'Epoch: {epoch:d}, Loss - Train: {train_loss:.7f}, Val MSE: {val_loss:.7f}, Val RMSE: {val_loss_rmse:.7f}')
    
    # --- Early Stopping Check ---
    improved = False
    if current_val_loss < best_val_loss:
        best_val_loss = current_val_loss
        improved = True

    if improved:
        patience_counter = 0 # Reset patience
        best_model_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch
        print(f'   -> Validation loss improved to {best_val_loss:.7f}. Saving model state.')
    else:
        patience_counter += 1
        print(f'   -> Validation metric did not improve. Patience: {patience_counter}/{patience}')

    if patience_counter >= patience:
        print(f'!!! Early stopping triggered after epoch {epoch}. Best epoch was {best_epoch} with Val Loss: {best_val_loss:.5f}')
        break # Exit the training loop

# --- After Training Loop ---
# Load the best model state found during training
if best_model_state:
    print("Loading best model state for final evaluation.")
    model.load_state_dict(best_model_state)
    # model.load_state_dict(torch.load('best_model.pth')) # If saved to disk
else:
    print("Warning: No best model state saved (perhaps training ended too early or validation loss never improved).")

# Final evaluation on the test set using the best model
print("Evaluating final model on Test Set...")
final_train_mse = evaluate(train_loader, model=model)
final_train_rmse = evaluate_rmse(train_loader, model=model)
final_val_mse = evaluate(val_loader, model=model)
final_val_rmse = evaluate_rmse(val_loader, model=model)
final_test_mse = evaluate(test_loader, model=model)
final_test_rmse = evaluate_rmse(test_loader, model=model)
print(f'Final Train Loss (best model): MSE {final_train_mse:.4f}, RMSE {final_train_rmse:.4f}')
print(f'Final Val Loss (best model): MSE {final_val_mse:.4f}, RMSE {final_val_rmse:.4f}')
print(f'Final Test Loss (best model): MSE {final_test_mse:.4f}, RMSE {final_test_rmse:.4f}')




### 6.4 Saving the model


In [ ]:
# Weights only (state_dict), not the whole object: loading it back needs the class definition
# from 3.2, which is the intended way to do this in PyTorch.

# Save model
# save model
save_path = MODEL_DIR / 'GCN_rep.pth'
torch.save(copy.deepcopy(model.state_dict()), save_path)

### 6.5 Training curves


In [ ]:
# Train and validation RMSE against epoch for this single model, with the best epoch marked.
# Compare the shape with the averaged cross-validation curves in 5.3.

# visualize the loss as the network trained
fig, ax = plt.subplots(figsize=(12,9),dpi=100) # Slightly larger figure for bigger fonts

for spine in ax.spines.values():
    spine.set_linewidth(2)  # Change 2 to desired thickness

epochs = list(range(1, len(train_losses) + 1)) # X-axis data
data = {
    'epoch': epochs,
    'train_mse': train_losses,
    'train_rmse': train_losses_rmse,
    'val_mse': val_losses,
    'val_rmse': val_losses_rmse,
}
data_df = pd.DataFrame(data)
output_filepath = FIG_DIR / 'GCN_rep-model_data.txt'
data_df.to_csv(output_filepath, index=False)

plt.plot(epochs,train_losses_rmse, c='b', label='Train', linewidth=4)
plt.plot(epochs,val_losses_rmse, c='orange', label='Validation', linewidth=4)

if best_epoch != -1: # Indicate best epoch
    plt.axvline(x=best_epoch, color='r', linestyle='--', linewidth=4) # label=f'Best Epoch ({best_epoch})',

# --- Fontsize Modifications ---
#plt.title('Representative GCN Model', fontsize=40, pad=15) # Increased title fontsize
ax.set_title("Representative RexGCN model", pad=10)
plt.xlabel('Epochs', fontsize=40) # Increased x-label fontsize
plt.ylabel('RMSE Loss (V)', fontsize=40) # Increased y-label fontsize


plt.tick_params(axis='both', which='major', width=2, labelsize=35) # Increased tick label size

num_epochs_run = len(val_losses_rmse)
plt.xlim(0, 200)

plt.ylim(0, 0.8)

# Tick Formatting
# Fixed number of major ticks to keep it clean
ax.xaxis.set_major_locator(MaxNLocator(nbins=6))
ax.yaxis.set_major_locator(MaxNLocator(nbins=5))

# Minor ticks (AutoMinorLocator(4) creates 3 minor ticks between majors)
ax.xaxis.set_minor_locator(AutoMinorLocator(4))
ax.yaxis.set_minor_locator(AutoMinorLocator(4))

ax.tick_params(direction='in', top=True, right=True, which='both')

# Legend
legend = ax.legend(loc='upper right', labelcolor='linecolor', frameon=True) #, fontsize=25)
frame = legend.get_frame()
frame.set_alpha(0.5)
frame.set_edgecolor('black')
frame.set_linewidth(1.0)



plt.tight_layout() # Adjust layout to prevent labels overlapping
plt.savefig(FIG_DIR / 'GCN_rep-model.png', transparent=True, bbox_inches='tight')
plt.show()




### 6.6 Parity plots and outliers

Predicted against true for all three splits. Points far from the diagonal are the complexes
the model cannot place; they are collected here so they can be looked at as chemistry rather
than as dots.


In [ ]:
# Predictions for train, validation and test, each as a parity plot, with complexes whose
# absolute error exceeds the threshold recorded for inspection.

import pprint

# --- Configuration ---
outlier_threshold = 0.8

# --- Set model to evaluation mode ---
model.eval()

# --- Initialize dictionary for original outlier info ---
original_outlier_info = {} # Changed from list to dict

# --- Initialize Plotting ---
print(f"Plotting results and finding outliers with threshold = {outlier_threshold}...")
fig, axes = plt.subplots(1, 3, figsize=(21, 10), dpi=100)


# fig.suptitle('Predicted vs. True Values - Outlier Visualization', fontsize=16)

# --- Process Train Data (Using non-shuffled loader for index mapping) ---
print("Processing Training data...")
y_true_train = []
y_pred_train = []

if 'train_loader_noshuffle' not in locals():
     print("Warning: 'train_loader_noshuffle' not found. Creating one. Ensure train_data exists.")
     train_loader_noshuffle = DataLoader(train_data, batch_size=args.batch_size, shuffle=False)

with torch.no_grad():
    for data in train_loader_noshuffle: # Use the non-shuffled loader here
        data = data.to(device)
        y_true_train.append(data.y)
        y_pred_train.append(model(data))

y_true_train = torch.cat(y_true_train, dim=0).cpu().numpy()
y_pred_train = torch.cat(y_pred_train, dim=0).cpu().numpy()

# Calculate R²
r2_train = round(r2_score(y_true_train, y_pred_train), 3)
print(f"R² Score Train set: {r2_train}")

# Identify outliers
error_train = np.abs(y_true_train - y_pred_train)
outlier_mask_train = error_train >= outlier_threshold 
non_outlier_mask_train = ~outlier_mask_train

# Find outlier indices within the split
split_outlier_indices_train = np.where(outlier_mask_train)[0]

# --- Store outlier info in the dictionary ---
num_outliers_train = 0
for split_idx in split_outlier_indices_train:
    original_idx = train_indices[split_idx]
    true_val = y_true_train[split_idx].item() # Use .item() to get scalar value
    pred_val = y_pred_train[split_idx].item() # Use .item() to get scalar value
    original_outlier_info[original_idx] = {'true': true_val, 'pred': pred_val}
    num_outliers_train += 1
print(f"Found and stored {num_outliers_train} outliers in train set.")

# Train Data Plot 
title_fontsize = 30
axlabel_fontsize = 30
ticklabel_fontsize = 25

axes[0].scatter(y_true_train, y_pred_train,
                color='royalblue', alpha=0.7, label='Predictions')
axes[0].plot([-2, 3], [-2, 3], 'k-', linewidth=1.5, label='Ideal (y=x)')
axes[0].set_xlabel("True Values", fontsize = axlabel_fontsize)
axes[0].set_ylabel("Predicted Values", fontsize = axlabel_fontsize)
axes[0].set_title("Training Data", fontsize = title_fontsize)
axes[0].legend(fontsize = 24, loc='upper left')
#axes[0].grid(True)
axes[0].set_xlim(-2, 3)
axes[0].set_xticks(np.arange(-2, 3.1, 1))
axes[0].set_ylim(-2, 3)
axes[0].set_yticks(np.arange(-2, 3.1, 1))
axes[0].tick_params(axis='both', which='major', labelsize=ticklabel_fontsize) # Increased tick label size
axes[0].set_aspect('equal', adjustable='box')
axes[0].text(0.95, 0.05, f"R² = {r2_train:.2f}", transform=axes[0].transAxes, ha='right', fontsize=35, bbox=dict(facecolor='white', alpha=0.8))

# --- Process Validation Data ---
print("Processing Validation data...")
y_true_val = []
y_pred_val = []
with torch.no_grad():
    for data in val_loader:
        data = data.to(device)
        y_true_val.append(data.y)
        y_pred_val.append(model(data))

y_true_val = torch.cat(y_true_val, dim=0).cpu().numpy()
y_pred_val = torch.cat(y_pred_val, dim=0).cpu().numpy()

# Calculate R²
r2_val = round(r2_score(y_true_val, y_pred_val), 3)
print(f"R² Score Val set: {r2_val}")

# Identify outliers
error_val = np.abs(y_true_val - y_pred_val)
outlier_mask_val = error_val >= outlier_threshold # Fixed HTML entity
non_outlier_mask_val = ~outlier_mask_val

# Find outlier indices within the split
split_outlier_indices_val = np.where(outlier_mask_val)[0]

# --- Store outlier info in the dictionary ---
num_outliers_val = 0
for split_idx in split_outlier_indices_val:
    original_idx = val_indices[split_idx]
    true_val = y_true_val[split_idx].item()
    pred_val = y_pred_val[split_idx].item()
    # Check if index already exists (e.g., if a point was somehow in multiple splits - unlikely but safe)
    if original_idx not in original_outlier_info:
         original_outlier_info[original_idx] = {'true': true_val, 'pred': pred_val}
         num_outliers_val += 1
    else:
         print(f"Warning: Index {original_idx} from validation set already found as outlier.")
print(f"Found and stored {num_outliers_val} new outliers in validation set.")


# Validation Data Plot (Plotting code remains the same)
axes[1].scatter(y_true_val, y_pred_val,
                color='royalblue', alpha=0.7, label='Non-Outliers')
axes[1].plot([-2, 3], [-2, 3], 'k-', linewidth=1.5, label='Ideal (y=x)')
axes[1].set_xlabel("True Values", fontsize = axlabel_fontsize)
#axes[1].set_ylabel("Predicted Values")
axes[1].set_title("Validation Data", fontsize = title_fontsize)
#axes[1].legend()
#axes[1].grid(True)
axes[1].set_xlim(-2, 3)
axes[1].set_xticks(np.arange(-2, 3.1, 1))
axes[1].set_ylim(-2, 3)
axes[1].set_yticks(np.arange(-2, 3.1, 1))
axes[1].tick_params(axis='both', which='major', labelsize=ticklabel_fontsize) # Increased tick label size
axes[1].set_aspect('equal', adjustable='box')
axes[1].text(0.95, 0.05, f"R² = {r2_val:.2f}", transform=axes[1].transAxes, ha='right', fontsize=35, bbox=dict(facecolor='white', alpha=0.8))

# --- Process Test Data ---
print("Processing Test data...")
y_true_test = []
y_pred_test = []
with torch.no_grad():
    for data in test_loader:
        data = data.to(device)
        y_true_test.append(data.y)
        y_pred_test.append(model(data))

y_true_test = torch.cat(y_true_test, dim=0).cpu().numpy()
y_pred_test = torch.cat(y_pred_test, dim=0).cpu().numpy()

# Calculate R² and RMSE (6.7 prints both on the parity plot)
r2_test = round(r2_score(y_true_test, y_pred_test), 3)
rmse_test = round(float(np.sqrt(mean_squared_error(y_true_test, y_pred_test))), 3)
print(f"R² Score Test set: {r2_test}")
print(f"RMSE Test set: {rmse_test} V")

# Identify outliers
error_test = np.abs(y_true_test - y_pred_test)
outlier_mask_test = error_test >= outlier_threshold # Fixed HTML entity
non_outlier_mask_test = ~outlier_mask_test

# Find outlier indices within the split
split_outlier_indices_test = np.where(outlier_mask_test)[0]

# --- Store outlier info in the dictionary ---
num_outliers_test = 0
for split_idx in split_outlier_indices_test:
    original_idx = test_indices[split_idx]
    true_val = y_true_test[split_idx].item()
    pred_val = y_pred_test[split_idx].item()
    if original_idx not in original_outlier_info:
         original_outlier_info[original_idx] = {'true': true_val, 'pred': pred_val}
         num_outliers_test += 1
    else:
         print(f"Warning: Index {original_idx} from test set already found as outlier.")
print(f"Found and stored {num_outliers_test} new outliers in test set.")

# Test Data Plot (Plotting code remains the same)
axes[2].scatter(y_true_test, y_pred_test,
                color='royalblue', alpha=0.7, label='Non-Outliers')
axes[2].plot([-2, 3], [-2, 3], 'k-', linewidth=1.5, label='Ideal (y=x)')

axes[2].set_xlabel("True Values", fontsize = axlabel_fontsize)
#axes[2].set_ylabel("Predicted Values")
axes[2].set_title("Test Data", fontsize = title_fontsize)
#axes[2].legend()
#axes[2].grid(True)
axes[2].set_xlim(-2, 3)
axes[2].set_xticks(np.arange(-2, 3.1, 1))
axes[2].set_ylim(-2, 3)
axes[2].set_yticks(np.arange(-2, 3.1, 1))
axes[2].tick_params(axis='both', which='major', labelsize=ticklabel_fontsize) # Increased tick label size
axes[2].set_aspect('equal', adjustable='box')
axes[2].text(0.95, 0.05, f"R² = {r2_test:.2f}", transform=axes[2].transAxes, ha='right', fontsize=35, bbox=dict(facecolor='white', alpha=0.8))


# --- Finalize and Show Results ---
# Sort the dictionary by key (original index) for consistent output
sorted_outlier_info = dict(sorted(original_outlier_info.items()))

print(f"\nTotal number of unique outliers found: {len(sorted_outlier_info)}")
print("Original indices and values of outliers (from graph_data):")

pprint.pprint(sorted_outlier_info, indent=2)

plt.savefig(FIG_DIR / 'GCN_trueVspred.png', transparent=True, bbox_inches='tight')

### 6.7 Test parity

The single figure that answers "how good is this model": the held-out test set, with $R^2$
and RMSE printed on the axes.


In [ ]:
# Publication version of the test-set parity plot.

# Plotting only the test parity

fig, ax = plt.subplots(figsize=(10, 10), dpi=100)

title_fontsize = 35
axlabel_fontsize = 30
ticklabel_fontsize = 25

ax.scatter(y_true_test,  y_pred_test,  color='red', alpha=0.7, marker='o', label='Test')

# Ideal line
ax.plot([-2, 3], [-2, 3], 'k--', linewidth=1.5) #, label='Ideal (y=x)')

# Labels, ticks, formatting
ax.set_xlabel("True Redox Potential (V)", fontsize=axlabel_fontsize)
ax.set_ylabel("Predicted Redox Potential (V)", fontsize=axlabel_fontsize)
ax.set_xlim(-2, 3)
ax.set_xticks(np.arange(-2, 3.1, 1))
ax.set_ylim(-2, 3)
ax.set_yticks(np.arange(-2, 3.1, 1))
ax.tick_params(axis='both', which='major', labelsize=ticklabel_fontsize)
ax.set_aspect('equal', adjustable='box')


# Print text on plot (in matching colors)

ax.text(0.52, 0.15, r"$R^{2}_{\text{test}}$=" + f"{r2_test:.2f}", transform=ax.transAxes, ha='left', fontsize=30, color='r') # , bbox=dict(facecolor='white', alpha=0.8)

ax.text(0.52, 0.05, r"$RMSE_{\text{test}}$=" + f"{rmse_test:.2f} V", transform=ax.transAxes, ha='left', fontsize=30, color='r') # , bbox=dict(facecolor='white', alpha=0.8)

# --------------------------------------------------
# Finalize
# --------------------------------------------------


plt.savefig(FIG_DIR / 'GCN_testonly_parity.png', transparent=True, bbox_inches='tight')
plt.show()


## Part 7 - Architectures that use the geometry

RexGCN reads the molecular graph: which atoms are bonded, and what kind of bond. It never
looks at `pos`, so two conformers of the same complex are identical to it.

Two other architectures in this project do use the coordinates, and both are available in
PyTorch Geometric:

### SchNet (continuous filters)

Builds its messages from *interatomic distances*, expanded in a basis of Gaussians and passed through continuous-filter convolutions. Neighbors are everything inside a cutoff radius, not just bonded atoms.

Uses distance-dependent continuous filters instead of fixed convolutions:

$$\mathbf{h}_i^{(l+1)} = \sum_{j: d_{ij} < \text{cutoff}} \text{filter}(d_{ij}) \odot \mathbf{h}_j^{(l)}$$

where $\text{filter}(d)$ is learned as a continuous function of distance via Gaussian basis expansion.

**When to use**:
- When you want smooth distance-dependent interactions
- Good for properties sensitive to atomic distances

### DimeNet++ (3D-aware)

Goes further and uses *angles* between triplets of atoms as well asdistances, through a spherical-harmonic basis. That makes it directional, which matters for coordination geometry, and considerably more expensive.

Directional Graph Neural Networks that use **angles** and **distances** directly:

$$\phi(d_{ij}, d_{jk}, \theta_{ijk}) = \text{basis expansion using RBF + spherical harmonics}$$

**Key insight**: doesn't construct a fixed graph; instead uses all atom pairs within a cutoff (8 Å) and refines connections based on 3D geometry.

**When to use**:
- When 3D structure is crucial
- When you have precise atomic coordinates
- For molecular property prediction (often better than GCN/GAT)

Neither is trained here: DimeNet++ and SchNet take hours to days on this dataset, well
past the length of a session. The cells below only construct the models, so the shape and the
parameter count can be compared with RexGCN.

Note that the same `Data` objects from Part 2 feed all three models. Because `GenGraphs`
attaches `z` and `pos`, nothing about the pipeline has to change to run them.


### Comparison table

| Model | Complexity | Memory | Speed | Best for |
|---|---|---|---|---|
| **GCN** | Low | Low | Fast | Prototyping, large graphs |
| **DimeNet++** | High | High | Slow | 3D geometry-critical properties |
| **SchNet** | Medium | Medium | Slow | Distance-dependent interactions |

In [ ]:
# DimeNet++ constructed with the settings used in the research notebook. Nothing is trained
# here; the point is the architecture and the parameter count.

from torch_geometric.nn.models import DimeNetPlusPlus


class TrainArgsDimeNetPP:
    def __init__(self,
                 # DimeNet++ specific hyperparameters
                 hidden_channels=128,
                 out_channels=1,         # Default for regression
                 num_blocks=4,
                 int_emb_size=64,   # Embedding size in interaction block
                 basis_emb_size=8,     # Embedding size for basis functions
                 out_emb_channels=256,
                 num_spherical=7,
                 num_radial=6,
                 cutoff=5.0,
                 envelope_exponent=5,
                 num_before_skip=1,
                 num_after_skip=2,
                 num_output_layers=3,

                 # General training args
                 batch_size=128,
                 init_lr=1e-4,
                 max_lr=1e-3,
                 final_lr=1e-5,
                 num_lrs=1,
                 warmup_epochs=2.0,
                 epochs=400,
                 patience=50):

        # Assign arguments to instance attributes
        # DimeNet++ specific
        self.hidden_channels = hidden_channels
        self.out_channels = out_channels
        self.num_blocks = num_blocks
        self.int_emb_size = int_emb_size
        self.basis_emb_size = basis_emb_size
        self.out_emb_channels = out_emb_channels
        self.num_spherical = num_spherical
        self.num_radial = num_radial
        self.cutoff = cutoff
        self.envelope_exponent = envelope_exponent
        self.num_before_skip = num_before_skip
        self.num_after_skip = num_after_skip
        self.num_output_layers = num_output_layers

        # General training args
        self.batch_size = batch_size
        self.init_lr = init_lr
        self.max_lr = max_lr
        self.final_lr = final_lr
        self.num_lrs = num_lrs
        self.warmup_epochs = warmup_epochs
        self.epochs = epochs
        self.patience = patience


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# train args
args_dimenet = TrainArgsDimeNetPP()

# Create the DimeNetPlusPlus model directly
model_dimenet = DimeNetPlusPlus(
    hidden_channels=args_dimenet.hidden_channels,
    out_channels=args_dimenet.out_channels,  # Should be 1 for regression
    num_blocks=args_dimenet.num_blocks,
    int_emb_size=args_dimenet.int_emb_size,
    basis_emb_size=args_dimenet.basis_emb_size,
    out_emb_channels=args_dimenet.out_emb_channels,
    num_spherical=args_dimenet.num_spherical,
    num_radial=args_dimenet.num_radial,
    cutoff=args_dimenet.cutoff,
    # max_num_neighbors=32, # Default is 32, can be omitted
    envelope_exponent=args_dimenet.envelope_exponent,
    num_before_skip=args_dimenet.num_before_skip,
    num_after_skip=args_dimenet.num_after_skip,
    num_output_layers=args_dimenet.num_output_layers,
    # act='swish', # Default is swish, can omit
    # output_initializer='zeros', # Default is zeros, can omit
).to(device)

print('Number of parameters (DimeNet++):',
      sum(p.numel() for p in model_dimenet.parameters() if p.requires_grad))


In [ ]:
# SchNet, same treatment: constructed only, so the parameter count can be compared.

from torch_geometric.nn import SchNet

class TrainArgsSchNet:
    def __init__(self,
                 # SchNet specific hyperparameters
                 hidden_channels=128,    # Hidden embedding size
                 num_filters=128,        # Number of filters (often same as hidden_channels)
                 num_interactions=6,     # Number of interaction blocks (like GNN layers)
                 num_gaussians=50,       # Number of Gaussians for distance expansion
                 cutoff=8.0,            # Cutoff distance for interactions
                 # Optional SchNet args from its __init__ if you need them:
                 # interaction_graph=None, # Custom graph generation function
                 # max_num_neighbors=32,   # If not using custom interaction_graph
                 readout='add',          # Aggregation for final output ('add' or 'mean')
                 dipole=False,           # Specific to dipole moment prediction
                 mean=None,              # For output standardization
                 std=None,               # For output standardization
                 atomref=None,           # Atomic reference energies

                 # General training args (kept the same as your DimeNetPP args)
                 batch_size=128,
                 init_lr=1e-4,
                 max_lr=1e-3,
                 final_lr=1e-5,
                 num_lrs=1,
                 warmup_epochs=2.0,
                 epochs=2000,
                 patience=100):

        # Assign arguments to instance attributes
        # SchNet specific
        self.hidden_channels = hidden_channels
        self.num_filters = num_filters
        self.num_interactions = num_interactions
        self.num_gaussians = num_gaussians
        self.cutoff = cutoff
        # self.interaction_graph = interaction_graph # Uncomment if using
        # self.max_num_neighbors = max_num_neighbors # Uncomment if using
        self.readout = readout
        self.dipole = dipole
        self.mean = mean
        self.std = std
        self.atomref = atomref

        # General training args
        self.batch_size = batch_size
        self.init_lr = init_lr
        self.max_lr = max_lr
        self.final_lr = final_lr
        self.num_lrs = num_lrs
        self.warmup_epochs = warmup_epochs
        self.epochs = epochs
        self.patience = patience


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# train args
args_schnet = TrainArgsSchNet()


# Create the SchNet model directly
model_schnet = SchNet(
    hidden_channels=args_schnet.hidden_channels,
    num_filters=args_schnet.num_filters,
    num_interactions=args_schnet.num_interactions,
    num_gaussians=args_schnet.num_gaussians,
    cutoff=args_schnet.cutoff,
    # Optional SchNet parameters (can use defaults or values from args):
    # max_num_neighbors=32, # Default, can be omitted or set in args
    readout=args_schnet.readout,   # e.g., 'add' or 'mean'
    dipole=args_schnet.dipole,     # Usually False for general regression
    mean=args_schnet.mean,         # Usually None unless doing specific QM9 tasks
    std=args_schnet.std,           # Usually None
    atomref=args_schnet.atomref    # Usually None unless doing specific QM9 tasks
    # Note: SchNet's out_channels is implicitly 1 for standard regression use
    # via its internal self.lin2 = Linear(hidden_channels // 2, 1)
    # before the readout.
).to(device)

print('Number of parameters (SchNet):',
      sum(p.numel() for p in model_schnet.parameters() if p.requires_grad))


### A note on running DimeNet++ elsewhere

In the research environment, constructing DimeNet++ failed inside PyG's spherical-harmonic
helper, which called `np.math.factorial`; that alias was removed from NumPy. The fix used
there was to patch the function in memory before building the model:

```python
import math
import numpy as np
from torch_geometric.nn.models import dimenet_utils

original_sph_harm_prefactor = dimenet_utils.sph_harm_prefactor

def corrected_sph_harm_prefactor(k, m):
    # Use math.factorial instead of np.math.factorial
    return ((2 * k + 1) * math.factorial(k - abs(m)) /
            (4 * np.pi * math.factorial(k + abs(m))))**0.5

dimenet_utils.sph_harm_prefactor = corrected_sph_harm_prefactor
print("Applied monkey patch to dimenet_utils.sph_harm_prefactor")
```

It is not needed with the versions installed for this workshop, so the cell above runs
without it. If you hit an `AttributeError` on `np.math` when you take this code elsewhere,
that is the fix, and it belongs before the model is constructed.


---

## Exercises

### Exercise 1 - Give RexGCN its graph-level features

Section 3.3 pointed out that the model ignores everything the pipeline computes at the graph
level. Put the fingerprint back.

1. `GenGraphs` already attaches `morgan_fp` to every `Data` object, of shape `[1, 64]` when
   called without a mask. Confirm that with `demo_graphs[0].morgan_fp.shape`, then check what
   a *batch* of them looks like: iterate one batch out of `train_loader` and print
   `batch.morgan_fp.shape`. PyG stacks graph-level attributes row-wise, so a batch of 128
   graphs gives `[128, 64]`, which lines up with `x_pooled` of shape `[128, 2*hidden_dim]`.
2. Add a `global_features_dim` argument back to `RexGCN.__init__`, store it, and widen the
   first fully connected layer: `fc_input_dim = (hidden_dim * 2) + global_features_dim`.
3. In `forward`, after the pooling line, concatenate instead of passing `x_pooled` straight
   through:

   ```python
   x = torch.cat([x_pooled, data.morgan_fp], dim=1)
   ```

4. Instantiate with `global_features_dim=64` and rerun the cross-validation from 5.2. Report
   mean and standard deviation of the validation RMSE with and without the fingerprint.
5. Three more graph-level fields are sitting in the `Data` objects: `metal_charge`,
   `metal_neigh_symbols` (92 wide) and `metal_neigh_charge`. The scalar ones need
   `.view(-1, 1)` before concatenation. Add them and adjust `global_features_dim` to match
   the total width. Does the metal environment help more than the fingerprint?
6. Argue, in two or three sentences, whether a fingerprint is a fair thing to give a GNN. The
   fingerprint encodes substructures the message passing could in principle discover for
   itself, in $L$ rounds, from the same graph.

### Exercise 2 - Hyperparameters, measured properly (required)

Change **one** hyperparameter in `TrainArgsGCN`, rerun 5.2, and report the mean and standard
deviation of the best validation RMSE across the folds. Suggestions, in rough order of how
much they usually matter:

| Parameter | Current | Try | What you are testing |
|---|---:|---|---|
| `num_conv_layers` | 3 | 2 or 5 | how many bonds of context the model needs |
| `hidden_dim` | 512 | 256 | whether the model is oversized for 1546 complexes |
| `dropout` | 0.1 | 0.3 | whether it is overfitting |
| `max_lr` | 1e-3 | 5e-4 | stability of the schedule |
| `batch_size` | 128 | 64 | noisier gradients, more updates per epoch |
| `norm_type` | `'layer'` | `'graph'` or `'batch'` | which normalisation suits variable-size graphs |

Then answer the question the exercise is really about: **is your change an improvement?** A
change counts only if the difference in the means is larger than the fold-to-fold spread. If
it is not, say so; reporting "no measurable difference" is a result.

Change one thing at a time. Two changes at once and you cannot attribute the outcome to
either.

### Exercise 3 - Error analysis

1. Take the outliers collected in 6.6 and pull the matching rows out of `final_df`.
2. Look at their SMILES and their ligand classes from Day 3 Part 5. Is there a class the
   model is systematically bad at?
3. Plot absolute error against number of atoms. Are larger complexes harder?
4. Plot a histogram of the residuals and report the skewness and kurtosis, as in Day 2
   section 1.2. A biased model has a residual distribution that is not centred on zero.

### Exercise 4 - Discussion

1. The GNN sees connectivity and bond types; the Random Forest of Day 3 saw counts and
   averages. Name a pair of complexes the GNN can tell apart and the Random Forest cannot.
2. RexGCN never looks at `pos`. Name a chemical situation where that loses information, and
   say which of the Part 7 architectures would recover it.
3. 1546 complexes is a small dataset for deep learning. What evidence in this notebook speaks
   to whether there is enough data?
4. If you had a week of GPU time, what would you run, and what would the result tell you?

---


---

## Part 8 - Lightning-talk structure

| Slide | Time | Contents |
|---|---:|---|
| 1. Title & problem | 30 s | Names, title, one-sentence problem statement |
| 2. The data | 45 s | Sample count, key features, one EDA figure (distribution or correlation) |
| 3. Classical ML results | 60 s | PCA: how many components capture $\geq 90\%$ variance? Clustering summary; RF/GPR test $\mathrm{RMSE}$ and $R^2$; top features |
| 4. GNN results | 60 s | Architecture (layers, hidden dim), cross-validation mean and spread, test parity plot; comparison to baselines |
| 5. Key insights & future work | 45 s | Structure–property takeaways, best model, next steps, impact |

Example title: "Predicting Iron Redox Potentials Using Graph Neural Networks to Accelerate Battery Material Discovery"

---

## Part 9 - Guiding questions

### 9.1 Scientific questions

1. What structural features of iron complexes most strongly influence redox potential ($E^0$)?
2. Are there distinct "families" of iron complexes in the dataset (from clustering)?
3. Do complexes with higher coordination numbers tend to have higher or lower redox potentials?

### 9.2 Data-science questions

1. Was the dataset large enough for deep learning? What evidence supports your answer?
2. Did the GNN outperform classical ML? If so, by how much? If not, why?
3. What was the most surprising finding in your exploratory data analysis?

### 9.3 Practical questions

1. If a battery engineer asked you to recommend the best model for screening new iron complexes, which would you recommend and why?
2. What are the limitations of your analysis? What assumptions did you make?
3. How would you validate these predictions experimentally?

---

## Part 10 - Presentation tips

- Keep it visual: prefer plots over tables of numbers.
- Tell a story: Problem → Data → Methods → Results → Impact.
- Be honest about limitations: acknowledging what didn't work shows scientific maturity.
- Practice timing: 5 minutes goes fast, so rehearse at least once.
- Engage the audience: ask a question or pose a challenge at the end.

---

## Part 11 - Assembling your presentation package

This section provides two small helper cells: (1) copy the key figures into a single presentation directory and (2) load and display the final results table. Both are small, robust helpers intended to work whether figures were saved to the repository directory or to `$SCRATCH`.


In [ ]:
# Copy key figures to a presentation directory (search multiple likely locations)
import os
import shutil
from pathlib import Path

from_ip = os.getcwd()
SCRATCH = os.path.expandvars('$SCRATCH') or None
CANDIDATE_ROOTS = [from_ip, str(REPO_ROOT), str(REPO_ROOT / 'notebooks'),
                   str(OUTPUT_DIR), str(FIG_DIR)]
if SCRATCH:
    CANDIDATE_ROOTS.append(SCRATCH)

pres_dir = os.path.join(SCRATCH or '.', 'presentation_figures')
Path(pres_dir).mkdir(parents=True, exist_ok=True)

figures_to_copy = [
    'iron_redox_potentials.png',
    'redox_distribution.png',
    'correlation_heatmap.png',
    'feature_vs_target_scatter.png',
    'pca_2d_projection.png',
    'elbow_silhouette.png',
    'kmeans_clusters.png',
    'rf_parity_plots.png',
    'rf_feature_importance.png',
    'gpr_results.png',
    'model_comparison.png',
    'GCN_kfold-cv.png',
    'GCN_rep-model.png',
    'GCN_testonly_parity.png',
]

copied = []
missing = []
for fig in figures_to_copy:
    found = False
    for root in CANDIDATE_ROOTS:
        src = os.path.join(root, fig)
        if os.path.exists(src):
            dst = os.path.join(pres_dir, fig)
            shutil.copy2(src, dst)
            copied.append((fig, src))
            found = True
            break
    if not found:
        missing.append(fig)

print(f'Copied {len(copied)} figures to: {pres_dir}')
if copied:
    print('Copied files:')
    for f, src in copied:
        print(f'  - {f}  (from {src})')
if missing:
    print('\nWarning: the following figures were not found:')
    for f in missing:
        print(f'  - {f}')


In [ ]:
# Load and display final results table
import os
import pandas as pd
from IPython.display import display

results_path = REPO_ROOT / 'final_results.csv'
if results_path.exists():
    results = pd.read_csv(results_path)
else:
    results = pd.DataFrame({
        'Model': ['Random Forest', 'GPR', 'Graph Neural Network'],
        'RMSE (Test)': ['-', '-', '-'],
        'R^2 (Test)': ['-', '-', '-'],
    })

print('=' * 70)
print('  FINAL RESULTS: Predicting Iron Redox Potentials')
print('  Fe-Redox-GNN Workshop - NERSC')
print('=' * 70)

display(results)

print('\nKey Takeaways:')
print('  1. Iron redox potentials are tunable by ligand environment')
print('  2. Structural features (bond lengths, coordination) correlate with $E^0$')
print('  3. GNNs can learn directly from molecular graphs')
print('  4. ML models provide significant speedups over DFT calculations')
print('\nBest Model: [Fill in based on your results]')
print('  Test RMSE: [Fill in]')
print('  Test R^2:   [Fill in]')


---

## Part 12 - Bridge to advanced research

Congratulations on completing the bootcamp! If you want to explore the production research pipeline behind this workshop, the `gnnredox/` repository contains a full end-to-end project built on the tmQM dataset and multiple GNN architectures (GCN, GAT, DimeNet++, SchNet). This section provides a concise bridge from the bootcamp material to that production codebase.

### 12.1 What comes next?

This notebook ran the real pipeline: the cleaned tmQM-derived complexes, the production
graph builder, the RexGCN model from the paper, and stratified cross-validation. What was
left out is the breadth. The `gnnredox/` repository carries the same work with more of
everything.

### 12.2 What is still missing

| Aspect | This workshop | Production (gnnredox/) |
|---|---|---|
| Data | 1546 cleaned Fe complexes | full tmQM (2267 complexes) and the preprocessing that gets there |
| Preprocessing | done for you on Day 1 | xyz2mol, metal disconnection, charge extraction, ligand consistency checks |
| Graph features | node + edge, graph-level computed but unused | fingerprints, SOAP, formal charges, 3D coordinates, all wired into the models |
| Models | RexGCN, plus SchNet and DimeNet++ constructed only | GCN, GAT, DimeNet++, SchNet, all trained and compared |
| Validation | 5-fold stratified CV on one architecture | 3-fold and 5-fold CV across every architecture |
| Interpretation | not covered | integrated gradients and saliency maps over atoms |
| Infrastructure | one notebook | helper modules, saved checkpoints, batch job scripts |

### 12.3 Getting started with gnnredox/

Follow these concrete steps to explore the production pipeline (assumes you will clone the repository locally):

1. Clone and install the repository:


```bash
git clone https://github.com/alvarovm/Fe-Redox-GNN.git
cd Fe-Redox-GNN
export PYTHONNOUSERSITE=1  # (on HPC systems)
conda create -p ./conenv python=3.11.9
conda activate ./conenv
pip install -r requirements.txt
# Install the matching PyTorch+PyG wheels for your system (see gnnredox/README.md)
python -m ipykernel install --user --name=fe-redox --display-name "Python (Fe-Redox)"
```


2. Explore the data with `Data_analysis.ipynb` (it shows dataset structure, filters, PCA/TSNE, and distributions).

3. Start with `Model_training/1b_GCN_base.ipynb`: it is the notebook that is closest to the bootcamp GCN but uses the real dataset and cross-validation.

### 12.4 Resources and further reading

- gnnredox README: https://github.com/alvarovm/Fe-Redox-GNN (setup and workflow)
- `fhb_helpers.py` and `fhb_xyz2mol_helpers.py` in gnnredox/ for advanced preprocessing
- Papers: Kipf & Welling (GCN), Gilmer et al. (MPNN), DimeNet/SchNet references for 3D-aware GNNs

---

### Exercise 5 - Prepare and deliver your lightning talk (required)

1. Slide 1: Title and one-sentence problem statement (include your name(s)).
2. Slide 2: One compelling EDA figure (distribution or correlation) and dataset summary (N samples, key features).
3. Slide 3: Classical ML summary: PCA components capturing $\geq 90\%$ variance, number of clusters, RF/GPR test $\mathrm{RMSE}$ and $R^2$, top features (Day 2 and Day 3 Part 1).
4. Slide 4: GNN summary: architecture (layers, hidden dim), best epoch, training curves, cross-validation mean and spread, test parity plot, and test $\mathrm{RMSE}$ and $R^2$.
5. Slide 5: Three concrete takeaways and one follow-up experiment or application.
6. Rehearse the talk once and confirm it fits within 5 minutes.

**Mentor checkpoint 11**: before the talks

- Slides prepared and figures embedded.
- Numbers on Slides 3 and 4 match the values printed in this notebook.
- Teams have rehearsed timing.
- Proceed only after confirmation.

---

## Continued learning and references

Graph Neural Networks:

- PyTorch Geometric documentation
- Stanford CS224W: Machine Learning with Graphs
- Gilmer et al., "Neural Message Passing for Quantum Chemistry" (2017)

Computational chemistry + ML:

- Open Catalyst Project
- Materials Project
- Xie & Grossman, "Crystal Graph Convolutional Neural Networks" (2018)

Iron redox batteries:

- ESS Inc.
- Form Energy
- Review: "Iron-based flow batteries" (Nature Energy)

HPC & scientific computing:

- NERSC training resources
- ALCF training resources

---

**Mentor checkpoint 12**: after the talks

- All teams have presented.
- Group discussion notes and follow-ups captured.
- Workshop wrap-up completed.
- Proceed only after confirmation.

---

## Summary

| Component | Tool / API | Key concept |
|---|---|---|
| Representation | PyTorch Geometric `Data`, `DataLoader` | atoms as nodes, bonds as edges, three levels of features |
| Graph building | `GenGraphs` | 119 node features, 10 edge features, graph-level fields |
| Model | `RexGCNConv` inside `RexGCN` | message passing that uses edge features, max + mean readout |
| Optimization | Adam, `NoamLR`, early stopping | warm-up then decay, keep the best epoch |
| Honest evaluation | stratified k-fold CV | report a mean and a spread, not one number |
| Geometry-aware models | SchNet, DimeNet++ | distances, and distances plus angles |

| Deliverable | What it is | Location |
|---|---|---|
| Figure bundle | presentation figures (PNG) | `output/figures/` |
| Model checkpoint | `GCN_rep.pth` | `output/saved_models/` |
| Cross-validation curve data | `GCN_kfold-cv_data.txt` | `output/figures/` |
| Baseline comparison | `baseline_results.csv` from Day 3 | `output/` |
| Presentation | 5-slide lightning talk | your slide deck |

---

## After the workshop

You have completed the Fe-Redox-GNN workshop. You now have hands-on experience with:

- Scientific computing on HPC systems
- Data wrangling and visualization for materials science
- Classical machine learning for property prediction
- Graph neural networks for molecular property prediction, from the graph builder to
  cross-validated results

These skills apply to computational science, materials informatics, and AI for science. Keep exploring and iterating on your models.
